# ***Industrial Vision SLM Benchmark***

## ***Notebook 03 — Construction et évaluation du moteur de recherche documentaire***

### **Objectif**

Construire et évaluer un moteur de recherche sémantique
à partir du corpus documentaire OpenCV préparé dans
le notebook 02.

### **Périmètre**

Le moteur exploitera :

- 55 chunks documentaires sélectionnés.
- Un modèle d'embeddings adapté aux questions en français
  et aux documents techniques en anglais.
- Un index vectoriel FAISS.
- Des métriques quantitatives d'évaluation du retrieval.

### **Méthodologie**

- Vérifier le corpus documentaire.
- Étudier les représentations vectorielles.
- Générer les embeddings des documents.
- Construire un index de recherche.
- Implémenter la recherche Top-k.
- Évaluer la pertinence des passages retrouvés.
- Analyser les erreurs et les limites du système.

### **Contraintes**

- Réutiliser le corpus du notebook 02 sans le modifier.
- Préserver les identifiants et les métadonnées des chunks.
- Réserver le jeu de test figé à l'évaluation finale.
- Ne pas modifier les résultats de la baseline.
- Documenter les paramètres et les choix expérimentaux.
- Privilégier une solution réalisable localement.

### **Résultat attendu**

Un moteur de recherche documentaire fonctionnel,
évalué et reproductible, prêt à être intégré au futur
pipeline RAG avec Qwen3-1.7B.

## **1. Initialisation et vérification du corpus de recherche**

### **Objectif**

Charger le corpus documentaire préparé précédemment
et vérifier son intégrité avant toute transformation
en représentations vectorielles.

### **Méthodologie**

1. Repérer la racine du projet.
2. Localiser le corpus documentaire filtré.
3. Charger le manifeste de sélection.
4. Vérifier l'empreinte SHA-256.
5. Charger les 55 chunks.
6. Contrôler leurs identifiants.
7. Examiner un premier exemple.

### **Résultats attendus**

- Corpus RAG chargé avec succès.
- 55 chunks disponibles en mémoire.
- Empreinte SHA-256 conforme au manifeste.
- Identifiants uniques.
- Premier chunk correctement affiché.

### **Point de vigilance**

Aucune modification ne doit être apportée
aux données produites par le notebook 02.

In [ ]:
import json
import hashlib
from pathlib import Path

# 1. Racine du projet
current_dir = Path.cwd().resolve()

if (current_dir / "data" / "corpus").is_dir():
    project_dir = current_dir
elif (current_dir.parent / "data" / "corpus").is_dir():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError("Impossible de trouver data/corpus/. Vérifier le réperroire de travail.")

# 2. Chemins des fichiers
corpus_dir = project_dir / "data" / "corpus"
chunks_path = corpus_dir / "chunks" / "retrieval_chunks.jsonl"
manifest_path = corpus_dir / "retrieval_manifest.json"

# 3. Vérif de la présence des fichiers
assert chunks_path.is_file(), f"Corpus introuvable : {chunks_path}"
assert manifest_path.is_file(), f"Manifeste introuvable : {manifest_path}"

# 4. Chargement du manifeste
with manifest_path.open("r", encoding='utf-8') as file:
    manifest = json.load(file)

# 5. Empreinte du corpus
corpus_bytes = chunks_path.read_bytes()
actual_checksum = hashlib.sha256(corpus_bytes).hexdigest()

# 6. Comparaison des empreintes (manifeste & corpus)
assert actual_checksum == manifest['retrieval_chunks_sha256'], "Le corpus ne correspond au manifeste."

# 7. Chargement des chunks depuis le JSONL
chunks = [
    json.loads(line) for line in corpus_bytes.decode('utf-8').splitlines() if line.strip()
]

# 8. Vérif de leur nombre et de leurs ids
assert len(chunks) == manifest['selected_count']

chunk_ids = [chunk['chunk_id'] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids))

# 9. Affichage d'un bilan et d'un exemple
print("[OK] Corpus RAG chargé et vérifié.")
print("Répertoire du projet :", project_dir)
print("Nombre de chunks :", len(chunks))
print("SHA-256 :", actual_checksum)

print("\n=== Exemple de chunk ===")

example = chunks[0]

print("ID :", example["chunk_id"])
print("Domaine :", example["topic"])
print("Section :", example["section"])
print("Longueur :", len(example["text"]))

print("\nAperçu :")
print(example["text"][:300])

## **2. Étude et génération des embeddings**

### **Objectif**

Étudier la représentation vectorielle des textes
avant de générer les embeddings du corpus OpenCV.

### **Méthodologie**

1. Comprendre la représentation d'un texte par un vecteur.
2. Étudier le produit scalaire et la norme euclidienne.
3. Implémenter la similarité cosinus.
4. Comparer des vecteurs sur un exemple contrôlé.
5. Choisir un modèle d'embeddings adapté au corpus.
6. Générer et valider les embeddings des 55 chunks.

### **Résultats attendus**

- Compréhension de la similarité vectorielle.
- Première fonction de comparaison opérationnelle.
- Modèle d'embeddings sélectionné et documenté.
- Embeddings du corpus générés et vérifiés.

### *2.1. Compréhension des représentations vectorielles*

**Objectif :** implémenter la similarité cosinus
sur des vecteurs numériques simples.

La similarité cosinus compare les directions
de deux vecteurs :

$$
\operatorname{sim}(u,v) =
\frac{\vec{u} \cdot \vec{v}}
{\lVert \vec{u} \rVert \, \lVert \vec{v} \rVert}
$$

Nous utilisons trois vecteurs fictifs :

- Question : `(1, 0)`
- Document A : `(0.8, 0.6)`
- Document B : `(0, 1)`

Le document A doit obtenir une similarité supérieure
à celle du document B.

**Point de vigilance :** cette expérience permet
de comprendre le calcul mathématique. Elle ne constitue
pas encore une expérience de recherche sémantique réelle.

In [ ]:
import math

# 1. Vecteurs fictifs
question_vector = [1.0, 0.0]

document_a = [0.8, 0.6]
document_b = [0.0, 1.0]

# 2. Fonction de similarité cosinus
def cosine_similarity(vector_a, vector_b):
    # Vérifier les dimensions des vecteurs
    assert len(vector_a) == len(vector_b), "Dimensions des vecteurs incompatibles"
    assert len(vector_a) > 0, "Vecteurs vides"

    # Produit scalaire
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))

    # Normes euclidiennes
    norm_a = math.sqrt(sum(value ** 2 for value in vector_a))
    norm_b = math.sqrt(sum(value ** 2 for value in vector_b))

    # La similarité cosinus n'est pas définie pour des vecteurs à norme nulle
    assert norm_a > 0 and norm_b > 0, "Un vecteur à norme nulle ne peut pas être comparé."

    # Appliquer la formule mathématique
    return dot_product / (norm_a * norm_b)

# 3. Comparaison de la question à chaque document
score_a = cosine_similarity(question_vector, document_a)
score_b = cosine_similarity(question_vector, document_b)

# 4. Affichage des résultats
print("=== Similarités cosinus ===")

print(f"Question / Document A : {score_a:.4f}")
print(f"Question / Document B : {score_b:.4f}")


# 5. Vérifier le comportement attendu.
assert math.isclose(score_a, 0.8, abs_tol=1e-9)
assert math.isclose(score_b, 0.0, abs_tol=1e-9)

assert score_a > score_b

print("\n[OK] Similarités cosinus validées.")
print("[OK] Le document A est le plus proche.")

### *2.2. Sélection et chargement du modèle d'embeddings*

**Objectif :** sélectionner et tester un modèle capable
de représenter les questions françaises et les documents
techniques anglais dans un même espace vectoriel.

**Modèle sélectionné :** `intfloat/multilingual-e5-small`

**Justification :**

- Prise en charge multilingue.
- Architecture relativement légère.
- Vecteurs de 384 dimensions.
- Adapté à la recherche de passages documentaires.
- Utilisable localement sur CPU.

**Convention d'encodage :**

- Questions : préfixe `query: `
- Documents : préfixe `passage: `

**Méthodologie :**

1. Charger le modèle préentraîné.
2. Construire une question française.
3. Construire deux passages anglais.
4. Générer leurs embeddings.
5. Vérifier leurs dimensions et leurs normes.
6. Comparer les similarités obtenues.

**Point de vigilance :**

Cette expérience constitue un premier test qualitatif.
Elle ne suffit pas à établir les performances du modèle.
Celles-ci seront mesurées dans la section 5.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Modèle sélectionné
model_name = "intfloat/multilingual-e5-small"

# 2. Chargement du modèle sur CPU (Téléchargement effectué lors du premier chargement)
embedding_model = SentenceTransformer(model_name, device="cpu")

# 3. Préparation d'une question en français
question = "query: Quelle opération morphologique permet de réduire les objets blancs dans un image binaire ?"

# 4. Préparation de deux documents en anglais
documents = [
    "passage: Erosion removes pixels at the boundaries of foreground objects in binary images.",
    "passage: Gaussian filtering smooths an image using a low_pass filter."
]

# 5. Génération des représentations vectorielles
question_embedding = embedding_model.encode([question], normalize_embeddings=True)
document_embeddings = embedding_model.encode(documents, normalize_embeddings=True)

# 6. Vérif des dimensions des embeddings
assert question_embedding.shape == (1, 384)
assert document_embeddings.shape == (2, 384)

# 7. Vérif de la normalisation des vecteurs
assert np.allclose(np.linalg.norm(question_embedding, axis=1), 1.0, atol=1e-5)
assert np.allclose(np.linalg.norm(document_embeddings, axis=1), 1.0, atol=1e-5)

# 8. Calcul des similarités
# Pour des vecteurs unitaires, le produit scalaire correspond à la similarité cosinus
similarities = (question_embedding @ document_embeddings.T)[0]

# 9. Affichage des résultats
print("=== Modèle d'embeddings ===")
print("Modèle :", model_name)
print("Dimension :", question_embedding.shape[1])
print("Vecteurs documents :", document_embeddings.shape[0])

print("\n=== Similarités ===")

for index, score in enumerate(similarities, start=1):
    print(f"Document {index} : {score:.4f}")

print("\n[OK] Embeddings générés et contrôlés.")

### *2.3. Génération et validation des embeddings du corpus*

**Objectif :** transformer les 55 chunks documentaires
OpenCV en représentations vectorielles exploitables
pour la recherche sémantique.

**Méthodologie :**

1. Récupérer les textes des 55 chunks.
2. Ajouter le préfixe `passage: ` attendu par E5.
3. Générer les embeddings par lots (*batch processing*).
4. Contrôler les dimensions de la matrice obtenue.
5. Vérifier la normalisation des vecteurs.
6. Conserver la correspondance entre chaque embedding
   et son identifiant documentaire.

**Configuration :**

- Modèle : `intfloat/multilingual-e5-small`
- Matériel : CPU
- Dimension : 384
- Nombre de documents : 55
- Taille des lots : 8
- Normalisation : activée

**Résultats attendus :**

- Une matrice de dimensions `(55, 384)`.
- Des vecteurs numériques finis.
- Des normes euclidiennes proches de 1.
- Des identifiants documentaires conservés.

**Point de vigilance :**

L'ordre des embeddings doit correspondre exactement
à l'ordre des chunks d'origine.

Une modification de cet ordre sans mise à jour
des identifiants provoquerait des erreurs
d'association entre résultats et documents.

In [ ]:
import numpy as np

# 1. Vérif du nombre de docs dispo
assert len(chunks) == 55, "Le corpus doit contenir 55 chunks."

# 2. Récup des identifiants documentaires
# Leur ordre doit rester identique à celui des textes
corpus_chunk_ids = [chunk["chunk_id"] for chunk in chunks]

# 3. Préparation des textes pour multilingual-E5
# Le préfixe 'passage:' identifie les documents
corpus_texts = ["passage: " + chunk["text"] for chunk in chunks]

# 4. Vérif des données avant encodage
assert len(corpus_texts) == len(corpus_chunk_ids)
assert len(set(corpus_chunk_ids)) == len(corpus_chunk_ids)
assert all(text.strip() != "passage:" for text in corpus_texts)

# 5. Génération des embeddings par lots de 8 docs
# La normalisation facilite ensiute ensuite le calcul de similarité cosinus
corpus_embeddings = embedding_model.encode(corpus_texts, batch_size=8, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# 6. Contrôler les dimensions obtenues
expected_dimension = 384
assert corpus_embeddings.shape == (len(chunks), expected_dimension)

# 7. Vérif que les valeurs sont finies
assert np.all(np.isfinite(corpus_embeddings))

# 8. Vérif de la normalisation des vecteurs
embedding_norms = np.linalg.norm(corpus_embeddings, axis=1)
assert np.allclose(embedding_norms, 1.0, atol=1e-5)

# 9. Afficher le bilan.
print("\n=== Embeddings du corpus ===")
print("Modèle :", model_name)
print("Chunks encodés :", len(corpus_texts))
print("Dimensions :", corpus_embeddings.shape)
print("Type numérique :", corpus_embeddings.dtype)

print("\n=== Vérification des normes ===")
print(f"Norme minimale : {embedding_norms.min():.6f}")
print(f"Norme maximale : {embedding_norms.max():.6f}")

# 10. Vérif de la correspondance documentaire
print("\n=== Exemples de correspondance ===")

for index in range(3):
    print(f"Ligne {index} : {corpus_chunk_ids[index]}")

print("\n[OK] Embeddings du corpus générés.")
print("[OK] Dimensions & normalisation validées.")
print("[OK] Correspondance documentaire conservée.")


### *2.4. Vérification des longueurs de tokenisation*

**Objectif :** vérifier que les 55 passages documentaires respectent la longueur maximale acceptée par le modèle d'embeddings.

**Méthodologie :**

1. Récupérer le tokenizer du modèle E5.
2. Tokeniser les textes sans troncature.
3. Mesurer le nombre de tokens par passage.
4. Identifier les éventuels dépassements.
5. Vérifier que tous les documents sont compatibles avec la longueur maximale du modèle.

**Point de vigilance :**

Un chunk limité à 1 500 caractères ne contient pas nécessairement moins de 512 tokens.

Cette vérification doit donc être effectuée indépendamment du contrôle de longueur réalisé dans le notebook 02.

In [ ]:
# 1. Récup du tokenizer du modèle d'embeddings
tokenizer = embedding_model.tokenizer

# 2. Récup de la longueur max du modèle
max_seq_length = embedding_model.max_seq_length

# 3. Tokenizer les docs sans troncature
tokenized_corpus = tokenizer(corpus_texts, add_special_tokens=True, truncation=False, padding=False, return_attention_mask=False)

# 4. Calcul de la longueur ed cahque séquence
token_lengths = [len(token_ids) for token_ids in tokenized_corpus["input_ids"]]

# 5. Identification des docs dépassant la limite
oversized_indices = [
    index for index, length in enumerate(token_lengths) if length > max_seq_length
]

# 6. Afficher le bilan.
print("=== Audit de tokenisation ===")

print("Modèle :", model_name)
print("Longueur maximale :", max_seq_length)
print("Documents analysés :", len(token_lengths))

print("\n=== Statistiques ===")

print("Minimum :", min(token_lengths))
print("Moyenne :", round(
    sum(token_lengths) / len(token_lengths), 1
))
print("Maximum :", max(token_lengths))

print("\nDocuments dépassant la limite :", len(oversized_indices))

# 7. Identification précise des dépassements
if oversized_indices:
    print("\n=== Documents à examiner ===")

    for index in oversized_indices:
        print(corpus_chunk_ids[index], ":", token_lengths[index], "tokens")

    print("\n[WARNING] Certains passages dépassent la longueur max du modèle.")
else:
    print("[OK] Aucun passage ne dépasse la longueur max du modèle.")

# 8. Vérif de la cohérence de l'audit
assert len(token_lengths) == len(corpus_chunk_ids)

print("[OK] Audit de tokenization terminé.")

### *2.5. Traitement des passages dépassant la limite du modèle*

**Objectif :** prendre en compte les passages documentaires
dont la tokenisation dépasse les 512 tokens autorisés.

**Problème identifié :**

Deux chunks dépassent la longueur maximale du modèle :

- `opencv_thresholding_chunk_007` : 586 tokens.
- `opencv_template_matching_chunk_004` : 590 tokens.

**Stratégie retenue :**

1. Conserver les 55 chunks documentaires originaux.
2. Découper temporairement les textes trop longs
   en fenêtres de tokens.
3. Encoder séparément chaque fenêtre.
4. Calculer une moyenne pondérée des embeddings obtenus.
5. Normaliser le vecteur résultant.
6. Remplacer uniquement les deux embeddings concernés.

**Résultat attendu :**

- 55 embeddings conservés.
- 384 dimensions par embedding.
- Aucun segment encodé au-delà de la limite du modèle.
- Correspondances documentaires inchangées.
- Corpus et manifestes du notebook 02 inchangés.

**Limite méthodologique :**

La moyenne de plusieurs embeddings ne reproduit pas
exactement l'encodage d'un texte complet.

Cette méthode constitue une approximation documentée,
qui pourra être comparée à un découpage plus fin
lors de l'évaluation du retrieval.

In [ ]:
import numpy as np

# 1. Capacité du modèle
max_tokens = embedding_model.max_seq_length

# Prévoir une marge pour le préfixe 'passage:' , les tokens spéciaux et les variations de tokenization
prefix = "passage: "
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))

window_size = max_tokens - special_tokens - prefix_tokens - 24
assert window_size > 0

# 2. Conservation de l'ordre des chunks déjà établi
assert corpus_embeddings.shape == (len(chunks), 384)

# 3. Correction uniquement des embeddings trop longs
for index in oversized_indices:
    chunk_id = corpus_chunk_ids[index]
    original_text = chunks[index]["text"]

    # Tokeniser le contenu sans le tronquer
    token_ids = tokenizer.encode(original_text, add_special_tokens=False, truncation=False)

    windows = []
    weights = []
    window_lengths = []

    # 4. Répartition des tokens dans plusieurs fenêtres
    for start in range(0, len(token_ids), window_size):
        segment_ids = token_ids[start:start + window_size]

        # Reconstruire le texte du segment
        segment_text = tokenizer.decode(segment_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

        # Ajouter le préfixe attendu par ES
        window_text = prefix + segment_text

        # Vérifier la longueur réellement encodée
        encoded_length = len(tokenizer.encode(window_text, add_special_tokens=True, truncation=False))
        assert encoded_length <= max_tokens, f"Fenêtre trop longue pour {chunk_id}"

        windows.append(window_text)
        weights.append(len(segment_ids))
        window_lengths.append(encoded_length)

    # Vérifier que tous les tokens du texte ont été répartis dans les fenêtres
    assert sum(weights) == len(token_ids)

    # 5. Encodage de toutes les fenêtres du chunk
    window_embeddings = embedding_model.encode(windows, batch_size=8, convert_to_numpy=True, normalize_embeddings=True)

    # 6. Combinaison des vecteurs par moyenne pondérée.
    # Chaque fenêtre contribue selon sa longueur
    combined_embedding = np.average(window_embeddings, axis=0, weights=weights)

    # 7. Normalisation du vecteur résultant
    combined_norm = np.linalg.norm(combined_embedding)
    assert combined_norm > 0

    combined_embedding = combined_embedding / combined_norm

    # 8. Remplacement uniquement de la ligne concernée
    corpus_embeddings[index] = combined_embedding
    print(f"[OK] {chunk_id} : {len(windows)} fenêtres (tokens par fenêtre : {window_lengths})")

# 9. Vérif de la matrice finale
assert corpus_embeddings.shape == (55, 384)
assert np.all(np.isfinite(corpus_embeddings))

final_norms = np.linalg.norm(corpus_embeddings, axis=1)
assert np.allclose(final_norms, 1.0, atol=1e-5)

# 10. Affichage du bilan
print("\n=== Bilan des corrections ===")

print("Chunks originaux :", len(chunks))
print("Embeddings finaux :", corpus_embeddings.shape)
print("Chunks corrigés :", len(oversized_indices))

print("\n[OK] Embeddings des passages longs corrigés.")
print("[OK] Dimensions et normalisation conservées.")
print("[OK] Corpus documentaire original inchangé.")


### *2.6. Sauvegarde reproductible des embeddings*

**Objectif :** conserver les représentations vectorielles
du corpus documentaire afin de pouvoir les réutiliser
sans recalculer les embeddings.

**Méthodologie :**

1. Vérifier une dernière fois la matrice d'embeddings.
2. Sauvegarder les vecteurs au format NumPy `.npy`.
3. Sauvegarder les identifiants documentaires dans
   leur ordre d'origine.
4. Enregistrer les paramètres de génération
   et les empreintes SHA-256 dans un manifeste.
5. Recharger les fichiers pour vérifier leur intégrité.
6. Éviter toute réécriture inutile des fichiers existants.

**Fichiers produits :**

- `outputs/retrieval/corpus_embeddings.npy`
- `outputs/retrieval/chunk_ids.json`
- `outputs/retrieval/embedding_manifest.json`

**Paramètres :**

- Modèle : `intfloat/multilingual-e5-small`.
- Dimension : 384.
- Normalisation : activée.
- Préfixe documentaire : `passage: `.
- Traitement des passages longs : moyenne pondérée
  des embeddings de fenêtres de tokens.

**Résultats attendus :**

- 55 embeddings sauvegardés.
- Correspondances documentaires conservées.
- Empreintes SHA-256 vérifiées.
- Rechargement identique aux données en mémoire.
- Exécution répétée sans modification inutile.

**Point de vigilance :**

Les embeddings dépendent du modèle utilisé,
de ses poids, des paramètres d'encodage
et du corpus documentaire.

Le manifeste permet de tracer ces paramètres,
mais ne remplace pas le contrôle des versions
du modèle et de l'environnement logiciel.

In [ ]:
import hashlib
import io
import json
from pathlib import Path

import numpy as np

# 1. Définition du répertoire de sauvegarde
output_dir = project_dir / "outputs" / "retrieval"
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Définition des chemins des trois fichiers.
embeddings_path = output_dir / "corpus_embeddings.npy"
ids_path = output_dir / "chunk_ids.json"
embedding_manifest_path = output_dir / "embedding_manifest.json"

# 3. Vérif des données avant leur sauvegarde
assert corpus_embeddings.shape == (55, 384)
assert corpus_embeddings.dtype == np.float32
assert len(corpus_chunk_ids) == 55
assert len(set(corpus_chunk_ids)) == 55
assert np.all(np.isfinite(corpus_embeddings))
assert np.allclose(np.linalg.norm(corpus_embeddings, axis=1), 1.0, atol=1e-5)

# 4. Vérif du corpus documentaire d'origine
corpus_checksum = hashlib.sha256(chunks_path.read_bytes()).hexdigest()
assert corpus_checksum == manifest["retrieval_chunks_sha256"]

# 5. Préparation du fichier NumPy en mémoire
buffer = io.BytesIO()
np.save(buffer, corpus_embeddings, allow_pickle=False)
embedding_bytes = buffer.getvalue()

# 6. Préparation des identifiants documentaires
ids_data = {
    "schema_version": 1,
    "chunk_ids": corpus_chunk_ids
}

ids_content = json.dumps(ids_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
ids_bytes = ids_content.encode("utf-8")

# 7. Calcul des empreintes des 2 fichiers
embeddings_checksum = hashlib.sha256(embedding_bytes).hexdigest()
ids_checksum = hashlib.sha256(ids_bytes).hexdigest()

# 8. Préparation du manifeste de sauvegarde
embedding_manifest = {
    "schema_version": 1,
    "model_id": model_name,
    "source_corpus_sha256": corpus_checksum,
    "document_count": len(corpus_chunk_ids),
    "embedding_dimension": corpus_embeddings.shape[1],
    "dtype": str(corpus_embeddings.dtype),
    "normalized": True,
    "document_prefix": "passage: ",
    "max_sequence_length": embedding_model.max_seq_length,
    "batch_size": 8,
    "long_chunk_strategy": "token_window_weighted_mean",
    "window_size": window_size,
    "corrected_chunk_ids": [corpus_chunk_ids[index] for index in oversized_indices],
    "embeddings_sha256": embeddings_checksum,
    "chunk_ids_sha256": ids_checksum
}

manifest_content = json.dumps(embedding_manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
embedding_manifest_bytes = manifest_content.encode("utf-8")

# 9. Regroupement des 3 fichiers à enregistrer
files_to_save = {
    embeddings_path: embedding_bytes,
    ids_path: ids_bytes,
    embedding_manifest_path: embedding_manifest_bytes
}

# 10. Eviter une sauvegarde partielle ou un écrasement
existing_files = [path.exists() for path in files_to_save]
if any(existing_files):
    assert all(existing_files), "Sauvegarde incomplète : certains fichiers existent déjà."

    for path, expected__bytes in files_to_save.items():
        assert path.read_bytes() == expected__bytes, f"Le fichier existant {path.name} diffère."

    print("[INFO] Embeddings déjà sauvegardés et à jour.")
else:
    for path, content, in files_to_save.items():
        path.write_bytes(content)
    print("[OK] Embeddings & métadonnées sauvegardés.")

# 11. Recharge de la matrice depuis le disque
loaded_embeddings = np.load(embeddings_path, allow_pickle=False)
with ids_path.open("r", encoding="utf-8") as file:
    loaded_ids = json.load(file)

with embedding_manifest_path.open("r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

# 12. Vérif du contenu rechargé
assert np.array_equal(loaded_embeddings, corpus_embeddings)
assert loaded_ids["chunk_ids"] == corpus_chunk_ids
assert loaded_manifest == embedding_manifest

# 13. Vérif des empreintes des fichiers écrits
assert hashlib.sha256(embeddings_path.read_bytes()).hexdigest() == embeddings_checksum
assert hashlib.sha256(ids_path.read_bytes()).hexdigest() == ids_checksum

# 14. Affichage du bilan
print("\n=== Sauvegarde des embeddings ===")

print("Répertoire :", output_dir.relative_to(project_dir))
print("Modèle :", loaded_manifest["model_id"])
print("Matrice :", loaded_embeddings.shape)
print("Type :", loaded_embeddings.dtype)

print("\nFichiers :")
for path in files_to_save:
    print("-", path.name)

print("\nSHA-256 des embeddings :")
print(embeddings_checksum)

print("\n[OK] Rechargement des embeddings validé.")
print("[OK] Identifiants documentaires conservés.")
print("[OK] Intégrité des fichiers vérifiée.")

## **3. Construction de l'index vectoriel**

### **Objectif**

Construire un système de recherche vectorielle
capable de retrouver les passages OpenCV les plus
similaires à une question.

### **Méthodologie**

1. Étudier les distances et similarités vectorielles.
2. Implémenter une recherche exacte avec NumPy.
3. Construire un index vectoriel FAISS.
4. Comparer les résultats de FAISS à ceux de NumPy.
5. Sauvegarder et recharger l'index.
6. Vérifier l'intégrité et la reproductibilité.

### **Résultats attendus**

- Classement des 55 chunks par similarité.
- Index FAISS fonctionnel.
- Résultats cohérents avec une recherche exacte NumPy.
- Correspondances documentaires conservées.
- Index sauvegardé et vérifié.

### **Point de vigilance**

La recherche de similarité ne garantit pas
la pertinence technique des passages retrouvés.

La qualité du retrieval sera évaluée séparément
dans la section 5.

### *3.1. Étude des distances et similarités*

**Objectif :** implémenter une recherche vectorielle
exacte avec NumPy avant l'utilisation de FAISS.

**Principe mathématique :**

Pour deux vecteurs normalisés, le produit scalaire
correspond à la similarité cosinus.

$$
\operatorname{sim}(u,v) = u \cdot v
$$

La distance euclidienne au carré vérifie :

$$
\|u-v\|^2 = 2 - 2(u \cdot v)
$$

Maximiser la similarité revient donc à minimiser
la distance euclidienne, lorsque les vecteurs
sont normalisés.

**Méthodologie :**

1. Encoder une question française.
2. Calculer sa similarité avec les 55 embeddings.
3. Classer les documents par score décroissant.
4. Afficher les trois passages les plus proches.
5. Vérifier la relation entre similarité cosinus
   et distance euclidienne au carré.

**Résultat attendu :**

Une première recherche Top-3 exacte, effectuée
sans index FAISS.

**Limite :**

Cette expérience sert de référence technique.
Elle ne constitue pas une évaluation complète
de la pertinence des passages retrouvés.

In [ ]:
import numpy as np

# 1. Définition d'une question en français
search_question = "Comment l'érosion morphologique modifie-t-elle les objets blancs d'une image binaire ?"

# 2. Préparation de la question pour le modèle d'embeddings
query_test = "query: " + search_question

# 3. Génération de l'embedding normalisé de la question
query_embedding = embedding_model.encode([query_test], convert_to_numpy=True, normalize_embeddings=True)
assert query_embedding.shape == (1, 384)

# 4. Récup des embeddings documentaires sauvegardés
document_vectors = loaded_embeddings
assert document_vectors.shape == (55,384)
assert loaded_ids["chunk_ids"] == corpus_chunk_ids

# 5. Similarités avec les 55 documents, chaque produit scalaire donnant un score
similarity_scores = document_vectors @ query_embedding[0]
assert similarity_scores.shape == (55,)

# 6. Distances euclidiennes au carré
squared_distances = np.sum((document_vectors - query_embedding[0]) ** 2, axis=1)

# 7. Vérif de la relation mathématique théorique
assert np.allclose(squared_distances, 2 - 2 * similarity_scores, atol=1e-5)

# 8. Classement des documents par similarité décroissante
top_k = 3

ranked_indices = np.argsort(-similarity_scores, kind="stable")[:top_k]

# 9. Affichage des résultats
print("=== Première recherche vectorielle ===")

print("Question :", search_question)
print("Documents disponibles :", len(document_vectors))
print("Top-k :", top_k)

print("\n=== Documents retrouvés ===")

for rank, index in enumerate(ranked_indices, start=1):

    chunk = chunks[index]
    score = similarity_scores[index]

    assert chunk["chunk_id"] == loaded_ids["chunk_ids"][index]

    print(f"\n--- Résultat {rank} ---")
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])
    print(f"Similarité : {score:.4f}")

    print("Aperçu :")
    print(chunk["text"][:250])

# 10. Confirmation de la recherche
assert len(ranked_indices) == top_k
assert np.all(np.isfinite(similarity_scores))

print("\n[OK] Recherche exacte NumPy terminée.")
print("[OK] Relation cosinus / distance vérifiée.")
print("[OK] Correspondances documentaires conservées.")

### *3.2. Création de l'index FAISS*

**Objectif :** construire un index vectoriel exact à partir des 55 embeddings documentaires.

**Bibliothèque :** FAISS (Facebook AI Similarity Search).

**Type d'index :** `IndexFlatIP`.

**Justification :**

- Recherche exhaustive sans approximation.
- Produit scalaire compatible avec la similarité cosinus
  pour des vecteurs normalisés.
- Adapté à notre corpus de petite taille.
- Permet une comparaison directe avec NumPy.

**Méthodologie :**

1. Charger FAISS.
2. Vérifier les dimensions et la normalisation.
3. Créer un index `IndexFlatIP`.
4. Ajouter les 55 embeddings documentaires.
5. Encoder la question déjà utilisée avec NumPy.
6. Effectuer une recherche Top-3.
7. Comparer les indices et les scores avec NumPy.
8. Vérifier les correspondances documentaires.

**Résultats attendus :**

- Un index contenant exactement 55 vecteurs.
- Trois résultats classés par similarité décroissante.
- Les mêmes documents que la recherche NumPy.
- Des scores identiques à la précision numérique près.

**Point de vigilance :**

L'index conserve les vecteurs et leurs positions,
mais pas directement les textes OpenCV.

La correspondance entre les positions et les
identifiants documentaires doit rester inchangée.


In [ ]:
import faiss
import numpy as np

# 1. Préparation des vecteurs documentaires
# FAISS attend des tableux numériques float32
document_vectors_faiss = np.ascontiguousarray(loaded_embeddings, dtype=np.float32)

# 2. Vérif de la matrice documentaire
document_count, embedding_dimension = document_vectors_faiss.shape
assert document_count == 55
assert embedding_dimension == 384
assert np.allclose(np.linalg.norm(document_vectors_faiss, axis=1), 1.0, atol=1e-5)

# 3. Création d'un index exact par produit scalaire
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Un index nouvellement créé ne contient aucun vecteur
assert faiss_index.ntotal == 0

# 4. Ajout des 55 embeddings à l'index
faiss_index.add(document_vectors_faiss)
assert faiss_index.ntotal == document_count

# 5. Préparation d'une recherche top-3
query_vector_faiss = np.ascontiguousarray(query_embedding, dtype=np.float32)
assert query_vector_faiss.shape == (1, 384)

# 6. Recherche Top-3
faiss_scores, faiss_indices = faiss_index.search(query_vector_faiss, top_k)

# 7. Vérif des dimensions des résultats
assert faiss_scores.shape == (1, top_k)
assert faiss_indices.shape == (1, top_k)
assert np.all((faiss_indices >= 0) & (faiss_indices < document_count))

# 8. Comparaison des classements FAISS & NumPy
assert np.array_equal(faiss_indices[0], ranked_indices), "Classement FAISS différent de NumPy"

# 9. Comparaison des scores de similarité
numpy_top_scores = similarity_scores[ranked_indices]
assert np.allclose(faiss_scores[0], numpy_top_scores, atol=1e-5), "Scores FAISS différents de NumPy."

# 10. Affichage du bilan de l'index
print("=== Index vectoriel FAISS ===")

print("Type :", type(faiss_index).__name__)
print("Dimensions :", faiss_index.d)
print("Vecteurs indexés :", faiss_index.ntotal)
print("Top-k :", top_k)

# 11. Affichage des documents retrouvés
print("\n=== Résultats FAISS ===")

for rank, (index, score) in enumerate(zip(faiss_indices[0], faiss_scores[0]), start=1):
    chunk = chunks[int(index)]
    assert chunk["chunk_id"] == corpus_chunk_ids[int(index)]

    print(f"\n--- Résultat {rank} ---")
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])
    print(f"Similarité : {score:.4f}")

# 12. Confirmation des vérifs
print("\n[OK] Index FAISS construit.")
print("[OK] Les 55 vecteurs sont indexés.")
print("[OK] Classement identique à NumPy.")
print("[OK] Scores de similarité validés.")
print("[OK] Correspondances documentaires conservées.")

### *3.3. Sauvegarde et rechargement de l'index vectoriel*

**Objectif :** sauvegarder l'index FAISS pour le réutiliser
dans de prochaines sessions sans reconstruire
systématiquement la structure vectorielle.

**Méthodologie :**

1. Vérifier les dimensions et le nombre de vecteurs.
2. Sérialiser l'index FAISS.
3. Calculer son empreinte SHA-256.
4. Construire un manifeste reliant l'index
   aux embeddings et aux identifiants documentaires.
5. Enregistrer les fichiers sans écrasement inutile.
6. Recharger l'index depuis le disque.
7. Vérifier les vecteurs et les résultats de recherche.
8. Contrôler l'idempotence de la sauvegarde.

**Fichiers produits :**

- `outputs/retrieval/faiss_index.bin`
- `outputs/retrieval/faiss_index_manifest.json`

**Configuration :**

- Index : `IndexFlatIP`.
- Nombre de vecteurs : 55.
- Dimension : 384.
- Métrique : produit scalaire.
- Embeddings normalisés : oui.
- Recherche : exacte, sans approximation.

**Résultats attendus :**

- Index sauvegardé et rechargé.
- 55 vecteurs conservés.
- Résultats Top-3 identiques avant et après sauvegarde.
- Correspondance avec les identifiants préservée.
- Empreintes SHA-256 vérifiées.

**Point de vigilance :**

Le chargement d'un index binaire doit être effectué
uniquement depuis un fichier fiable et contrôlé.

Les fichiers FAISS ne doivent pas être chargés
depuis des sources inconnues ou non fiables.

In [ ]:
import hashlib
import json

import faiss
import numpy as np


# 1. Définition des chemins de sauvegarde
index_path = output_dir / "faiss_index.bin"
index_manifest_path = output_dir / "faiss_index_manifest.json"

# 2. Vérif de l'index avant sa sauvegarde
assert isinstance(faiss_index, faiss.IndexFlatIP)

assert faiss_index.ntotal == 55
assert faiss_index.d == 384

assert loaded_embeddings.shape == (55, 384)
assert loaded_ids["chunk_ids"] == corpus_chunk_ids

# 3. Vérif des fichiers dont dépend l'index
embeddings_checksum = hashlib.sha256(embeddings_path.read_bytes()).hexdigest()
ids_checksum = hashlib.sha256(ids_path.read_bytes()).hexdigest()

assert embeddings_checksum == loaded_manifest["embeddings_sha256"]
assert ids_checksum == loaded_manifest["chunk_ids_sha256"]

# 4. Sérialisation de l'index FAISS en mémoire
# On apprête les octets avant d'écrire sur le disque
index_bytes = faiss.serialize_index(faiss_index).tobytes()

# 5. Empreinte de l'index
index_checksum = hashlib.sha256(index_bytes).hexdigest()

# 6. Construction du manifeste
faiss_manifest = {
    "schema_version": 1,
    "index_type": "IndexFlatIP",
    "metric": "inner_product",
    "exact_search": True,
    "normalized_embeddings": True,
    "document_count": int(faiss_index.ntotal),
    "embedding_dimension": int(faiss_index.d),
    "model_id": loaded_manifest["model_id"],
    "source_embeddings_sha256": embeddings_checksum,
    "source_chunk_ids_sha256": ids_checksum,
    "source_corpus_sha256": loaded_manifest["source_corpus_sha256"],
    "index_sha256": index_checksum
}

# 7. Préparation du contenu JSON du manifeste
manifest_content = json.dumps(faiss_manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
manifest_bytes = manifest_content.encode("utf-8")

# 8. Regroupement des fichiers à sauvegarder
index_files = {
    index_path: index_bytes,
    index_manifest_path: manifest_bytes
}

# 9. Vérif des fichiers déjà présents
existing_files = [path.exists() for path in index_files]
if any(existing_files):
    assert all(existing_files), "Sauvegarde incomplète de l'index FAISS."

    for path, expected__bytes in index_files.items():
        assert path.read_bytes() == expected__bytes, f"Fichier existant différent : {path.name}"

    print("[INFO] Index FAISS déjà sauvegardé et à jour.")
else:
    for path, content in index_files.items():
        path.write_bytes(content)

    print("[OK] Index FAISS & manifeste sauvegardés.")

# 10. Vérif de l'empreinte du fichier enregistré
assert hashlib.sha256(index_path.read_bytes()).hexdigest() == index_checksum

# 11. Recharge de l'index depuis le disque
# Ne charger  que des fichiers locaux fiables
saved_index_bytes = index_path.read_bytes()

# Vérifier l'intégrité avant la désérialisation
assert hashlib.sha256(saved_index_bytes).hexdigest() == index_checksum

# Transformer les octets en tableau NumPy uint8
serialized_index = np.frombuffer(saved_index_bytes, dtype=np.uint8).copy()

# Reconstruire l'index sans passer son chemin à FAISS
loaded_faiss_index = faiss.deserialize_index(serialized_index)

# 12. Vérif de sa structure
assert isinstance(loaded_faiss_index, faiss.IndexFlatIP)
assert loaded_faiss_index.ntotal == 55
assert loaded_faiss_index.d == 384

# 13. Vérif des vecteurs rechargés
reconstructed_vectors = loaded_faiss_index.reconstruct_n(0, loaded_faiss_index.ntotal)
assert np.array_equal(reconstructed_vectors, loaded_embeddings)

# 14. Remake de la même recherche
loaded_scores, loaded_indices = loaded_faiss_index.search(query_vector_faiss, top_k)

# 15. Comparaison des résultats avant & après rechargement
assert np.array_equal(loaded_indices, faiss_indices)
assert np.allclose(loaded_scores, faiss_scores, atol=1e-6)

# 16. Vérif du manifeste rechargé
with index_manifest_path.open("r", encoding="utf-8") as file:
    saved_faiss_manifest = json.load(file)

assert saved_faiss_manifest == faiss_manifest

# 17. Affichage du bilan
print("\n=== Sauvegarde de l'index FAISS ===")

print("Type :", saved_faiss_manifest["index_type"])
print("Dimensions :", loaded_faiss_index.d)
print("Vecteurs :", loaded_faiss_index.ntotal)
print("Recherche exacte :", saved_faiss_manifest["exact_search"])

print("\nFichiers :")
print("-", index_path.name)
print("-", index_manifest_path.name)

print("\nSHA-256 de l'index :")
print(index_checksum)

print("\n[OK] Index FAISS rechargé.")
print("[OK] Vecteurs rechargés sans modification.")
print("[OK] Classement et scores conservés.")
print("[OK] Intégrité et métadonnées vérifiées.")

## **4. Implémentation de la recherche documentaire**

### **Objectif**

Transformer notre index FAISS en un moteur de recherche documentaire réutilisable.

### **Méthodologie**

1. Définir une fonction de recherche sémantique.
2. Encoder automatiquement chaque question.
3. Interroger l'index FAISS avec un Top-k configurable.
4. Associer les vecteurs retrouvés aux documents OpenCV.
5. Structurer les résultats et leurs scores.
6. Gérer les paramètres invalides.
7. Préparer l'intégration des métadonnées de provenance.

### **Résultats attendus**

- Une fonction de recherche réutilisable.
- Des résultats structurés et classés.
- Des textes associés aux bons identifiants.
- Des paramètres contrôlés.
- Une recherche fonctionnant sur plusieurs questions.

### **Point de vigilance**

Un score de similarité élevé ne constitue pas une preuve d'exactitude scientifique.

La pertinence des résultats sera évaluée indépendamment dans la section 5.


### *4.1. Première fonction de recherche sémantique*

**Objectif :** généraliser la recherche FAISS dans une fonction `retrieve_documents()`.

**Paramètres :**

- `question` : question de l'utilisateur.
- `top_k` : nombre de passages à retrouver.

**Fonctionnement :**

1. Vérifier les paramètres.
2. Ajouter le préfixe `query: ` à la question.
3. Générer et normaliser son embedding.
4. Effectuer la recherche dans FAISS.
5. Récupérer les chunks correspondant aux indices.
6. Retourner une liste de résultats structurés.

**Champs de chaque résultat :**

- `rank` : position dans le classement.
- `index` : position dans la matrice documentaire.
- `chunk_id` : identifiant du passage.
- `topic` : domaine technique.
- `section` : section documentaire.
- `score` : similarité cosinus.
- `text` : contenu du passage.

**Résultat attendu :**

La fonction doit reproduire les résultats de notre recherche précédente lorsque nous utilisons la même question et `top_k=3`.

**Point de vigilance :**

Les indices retournés par FAISS doivent correspondre à l'ordre des identifiants et des chunks enregistrés.


In [ ]:
import numpy as np
# 1. Correspondance entre l'index et les documents avant toute recherche
assert loaded_faiss_index.ntotal == len(chunks)
assert loaded_ids["chunk_ids"] == [chunk["chunk_id"] for chunk in chunks]
assert model_name == loaded_manifest["model_id"]

# 2. Fonction de recherche réutilisable
def retrieve_documents(question, top_k=3):
    # Vérif que la question est une chaîne non vide
    if not isinstance(question, str) or not question.strip():
        raise ValueError("La question doit être une chaîne non vide.")

    # Vérif que Top-k est un entier valide
    if type(top_k) is not int or not 1 <= top_k <= loaded_faiss_index.ntotal:
        raise ValueError(f"top_k doit être un entier compris entre 1 et {loaded_faiss_index.ntotal}.")

    # 3. Ajout du préfixe attendu par E5
    query_text = "query: " + question.strip()

    # 4. Génération de l'embedding normalisé de la question
    query_vector = embedding_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)

    # 5. Préparation des données pour FAISS
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    assert query_vector.shape == (1, loaded_faiss_index.d)
    assert np.all(np.isfinite(query_vector))

    # 6. Recherche dans l'index
    scores, indices = loaded_faiss_index.search(query_vector, top_k)

    # 7. Construction d'une liste de résultats structurés
    results = []

    for rank, (index, score) in enumerate(zip(indices[0], scores[0]), start=1):
        index = int(index)
        score = float(score)

        assert 0 <= index < len(chunks)
        assert np.isfinite(score)

        chunk = chunks[index]

        # Vérifier l'association entre le vecteur et le doc correspondant
        assert chunk["chunk_id"] ==loaded_ids["chunk_ids"][index]

        results.append({
            "rank": rank,
            "index": index,
            "chunk_id": chunk["chunk_id"],
            "topic": chunk["topic"],
            "section": chunk["section"],
            "score": score,
            "text": chunk["text"]
        })

    return results

# 8. Test sur la même question qu'en section 3
test_results = retrieve_documents(question=search_question, top_k=3)

# 9. Vérif de la structure et du classement
assert len(test_results) == 3
assert len({result["chunk_id"] for result in test_results}) == 3
assert all(test_results[i]["score"] >= test_results[i+1]["score"] for i in range(len(test_results) - 1))

# Comparaison avec le classement FAISS de la section 3.2
assert [result["index"] for result in test_results] == faiss_indices[0].tolist()

# 10. Afficher les résultats.
print("=== Fonction de recherche documentaire ===")

print("Question :", search_question)
print("Nombre de résultats :", len(test_results))

for result in test_results:

    print(f"\n--- Résultat {result['rank']} ---")
    print("ID :", result["chunk_id"])
    print("Domaine :", result["topic"])
    print("Section :", result["section"])
    print(f"Similarité : {result['score']:.4f}")

    print("Aperçu :")
    print(result["text"][:200])


print("\n[OK] Fonction de recherche opérationnelle.")
print("[OK] Classement FAISS reproduit.")
print("[OK] Résultats documentaires structurés.")


### *4.2. Recherche Top-k paramétrable sur plusieurs questions*

**Objectif :** vérifier que le moteur documentaire peut traiter plusieurs questions avec différentes valeurs du paramètre `top_k`.

**Méthodologie :**

1. Préparer trois questions techniques exploratoires.
2. Effectuer les recherches avec k = 1, 3 et 5.
3. Vérifier le nombre de documents retournés.
4. Contrôler l'unicité des identifiants.
5. Vérifier le classement décroissant des scores.
6. Comparer la stabilité des résultats entre les différentes valeurs de k.
7. Examiner les domaines et sections retrouvés.

**Questions explorées :**

- Morphologie : érosion d'une image binaire.
- Contours : filtrage gaussien et méthode de Canny.
- Correspondance de gabarit : template matching.

**Résultats attendus :**

- Neuf recherches réalisées.
- Des résultats Top-1, Top-3 et Top-5.
- Des classements cohérents entre les valeurs de k.
- Des métadonnées documentaires conservées.

**Point de vigilance :**

Cette exploration permet de vérifier le comportement technique du système et d'observer ses résultats.

Elle ne permet pas encore de choisir objectivement la meilleure valeur de k.

Cette sélection reposera sur les métriques définies lors de l'évaluation du retrieval.

In [ ]:
# 1. Préparation de 3 questions techniques exploratoires
exploratory_questions = {
    "morphology": "Comment l'érosion morphologique modifie-t-elle les objets blancs d'une image binaire ?",
    "edges": "Pourquoi appliquer un filtre gaussin avant la détection de contours avec Canny ?",
    "template_matching": "Comment retrouver un objet dans une image en utilisant la correspondance par gabarit ?"
}

# 2. Définition des valeurs de k à comparer
top_k_values = [1, 3, 5]

# 3. Conservation des résultats des différentes recherches
exploratory_results = {}

# 4. Parcours des questions
for topic, question in exploratory_questions.items():
    print(f"\n=== Domaine : {topic} ===")
    print("Quesion :", question)

    results_by_k = {}

    # 5. Recherches Top-k
    for k in top_k_values:
        results = retrieve_documents(question=question, top_k=k)
        results_by_k[k] = results

        # Vérif du nombre de résultats
        assert len(results) == k  

        # Vérif de l'unicité des identifiants
        result_ids = [result["chunk_id"] for result in results]
        assert len(result_ids) == len(set(result_ids))

        # Vérif de la décroissance du classement
        result_scores = [result["score"] for result in results]
        assert result_scores == sorted(result_scores, reverse=True)

        print(f"\n[OK] Recherche Top-{k} : {len(results)} résultats")

    # 6. Vérif de la stabilité des premiers résultats
    ids_top_1 = [result["chunk_id"] for result in results_by_k[1]]
    ids_top_3 = [result["chunk_id"] for result in results_by_k[3]]
    ids_top_5 = [result["chunk_id"] for result in results_by_k[5]]

    assert ids_top_1 == ids_top_3[:1]
    assert ids_top_3 == ids_top_5[:3]

    # 7. Affichage des 5 premiers résultats
    print("\n--- Classement Top-5 ---")

    for result in results_by_k[5]:

        print(
            f"{result['rank']}. "
            f"{result['chunk_id']} | "
            f"{result['topic']} | "
            f"{result['section']} | "
            f"score={result['score']:.4f}"
        )

    # 8. Conservation des résultats de la question.
    exploratory_results[topic] = results_by_k

    print("\n[OK] Classement stable entre k=1, k=3 et k=5.")


# 9. Vérif du bilan global.
assert len(exploratory_results) == 3
assert all(set(results_by_k) == set(top_k_values) for results_by_k in exploratory_results.values())

# 10. Affichage du bilan de l'expérience.
print("\n=== Bilan des recherches ===")

print("Questions explorées :", len(exploratory_questions))
print("Valeurs de k :", top_k_values)
print("Recherches réalisées :", len(exploratory_questions) * len(top_k_values))

print("\n[OK] Recherches multi-questions validées.")
print("[OK] Résultats Top-k cohérents.")
print("[OK] Correspondances documentaires conservées.")

### *4.3. Gestion des métadonnées et traçabilité des sources*

**Objectif :** enrichir les résultats du moteur
de recherche avec les informations de provenance
des documents OpenCV.

**Métadonnées exploitées :**

- Identifiant du chunk.
- Identifiant du document source.
- Titre de la documentation.
- Domaine technique.
- Section du document.
- Position de la section et du passage.
- URL officielle OpenCV.
- Score de similarité.
- Référence de citation `[S1]`, `[S2]`, etc.

**Méthodologie :**

1. Charger le registre des sources documentaires.
2. Vérifier la correspondance entre les sources et les métadonnées conservées dans les chunks.
3. Enrichir les résultats de recherche existants.
4. Générer des références de citation.
5. Vérifier la cohérence des identifiants et des URLs.
6. Afficher les résultats avec leur provenance.

**Résultats attendus :**

- Résultats documentaires traçables.
- Liens vers les documentations OpenCV originales.
- Références de citation correctement attribuées.
- Correspondances entre chunks et sources validées.
- Aucun changement des scores ou du classement FAISS.

**Point de vigilance :**

Une source correctement identifiée ne garantit pas qu'elle répond à la question.

La pertinence documentaire devra être vérifiée indépendamment pendant l'évaluation du retrieval.

In [ ]:
import json

# 1. Registre doucumentaire
source_registry_path = corpus_dir / "source_registry.json"
assert source_registry_path.is_file(), "Registre documentaire introuvable"

with source_registry_path.open('r', encoding="utf-8") as file:
    source_registry = json.load(file)

# 2. Association des sources à leurs identifiants
sources_by_id = {source["source_id"]: source for source in source_registry["sources"]}
assert len(sources_by_id) == len(source_registry["sources"])
assert len(sources_by_id) == 7

# 3. Vérif des métadonnées de tous les chunks
for chunk in chunks:
    document_id = chunk["document_id"]
    assert document_id in sources_by_id

    source = sources_by_id[document_id]
    assert chunk["document_title"] == source["title"]
    assert chunk["source_url"] == source["source_url"]
    assert chunk["topic"] == source["topic"]

print("[OK] Correspondance entre les 55 chunks et les 7 sources vérifiée.")


# 4. Fonction de recherche enrichie avec les sources
def retrieve_with_sources(question, top_k=3):
    # Réutiliser la fonction de recherche existante.
    base_results = retrieve_documents(question=question, top_k=top_k)

    enriched_results = []

    # 5. Enrichissement de chaque résultat
    for result in base_results:
        index = result["index"]
        chunk = chunks[index]
        source_id = chunk["document_id"]
        source = sources_by_id[source_id]

        # Vérifier la correspondance documentaire.
        assert result["chunk_id"] == chunk["chunk_id"]
        assert source_id == source["source_id"]
        assert chunk["source_url"] == source["source_url"]

        # 6. Ajout des métadonnées de provenance
        enriched_result = {
            **result,
            "source_id": source_id,
            "document_title": source["title"],
            "source_url": source["source_url"],
            "section_index": chunk["section_index"],
            "part_index": chunk["part_index"],
            "citation": f"[S{result['rank']}]"
        }

        enriched_results.append(enriched_result)

    return enriched_results

# 7. Test avec la question de template matching
source_results = retrieve_with_sources(question=exploratory_questions["template_matching"], top_k=3)

# 8. Vérif du nombre et de l'ordre des résultats
assert len(source_results) == 3
assert [result["chunk_id"] for result in source_results] == [result["chunk_id"] for result in exploratory_results["template_matching"][3]]

# 9. Vérif des références générées
assert [result["citation"] for result in source_results] == ["[S1]", "[S2]", "[S3]"]

# 10. Vérif des liens documentaires.
assert all(result["source_url"].startswith("https://") for result in source_results)

# 11. Affichage des résultats avec leurs sources.
print("\n=== Recherche avec traçabilité ===")

for result in source_results:

    print(f"\n--- Source {result['citation']} ---")

    print("Chunk :", result["chunk_id"])
    print("Document :", result["document_title"])
    print("Domaine :", result["topic"])
    print("Section :", result["section"])

    print(f"Similarité : {result['score']:.4f}")

    print("URL :", result["source_url"])

    print("Aperçu :")
    print(result["text"][:180])

print("\n[OK] Métadonnées documentaires vérifiées")

print("[OK] Sources et URLs conservées.")
print("[OK] Références de citation générées.")
print("[OK] Classement FAISS inchangé.")

## **5. Évaluation de la qualité du retrieval**

### **Objectif**

Mesurer quantitativement la capacité du moteur
de recherche à retrouver les passages documentaires
réellement pertinents pour une question technique.

### **Méthodologie**

1. Définir un protocole d'évaluation indépendant des scores de similarité.
2. Réutiliser les questions du jeu de développement.
3. Construire des jugements de pertinence documentaires.
4. Calculer Precision@k, Recall@k et MRR.
5. Comparer les résultats pour différentes valeurs de k.
6. Identifier les erreurs de classement.
7. Documenter les limites de l'évaluation.

### **Résultats attendus**

- Un protocole d'annotation explicite.
- Des jugements de pertinence traçables.
- Des métriques quantitatives reproductibles.
- Une analyse par question et par domaine.
- Une référence pour comparer les futurs retrievers.

### **Contraintes**

- Ne pas modifier le dataset d'évaluation figé.
- Utiliser le jeu de développement pour les ajustements.
- Préserver le jeu de test pour l'évaluation finale.
- Ne pas déduire la pertinence des seuls scores FAISS.
- Documenter les questions sans passage pertinent
  dans le corpus disponible.

### **Point de vigilance**

La qualité du retrieval et la qualité de génération
de Qwen3-1.7B doivent être évaluées séparément.

### *5.1. Construction du protocole d'évaluation*

**Objectif :** examiner le jeu de développement et préparer l'annotation des passages documentaires.

**Méthodologie :**

1. Charger le dataset d'évaluation existant.
2. Vérifier sa structure et son nombre de questions.
3. Séparer les ensembles développement et test.
4. Examiner uniquement les questions de développement.
5. Préparer les futures associations question-chunk.
6. Définir les règles de jugement de pertinence.

**Échelle de pertinence :**

- 0 : non pertinent.
- 1 : partiellement pertinent.
- 2 : directement pertinent.

**Résultat attendu :**

- 50 questions reconnues.
- 10 questions de développement identifiées.
- 40 questions de test conservées à l'écart.
- Aucune modification des fichiers d'évaluation.

**Point de vigilance :**

L'URL de référence d'une question peut aider à identifier une source pertinente.

Cependant, partager une URL ou un domaine ne suffit pas pour garantir la pertinence de chaque chunk issu de cette source.

In [ ]:
import json
from collections import Counter

# 1. Chemin du dataset figé
evaluation_path = project_dir / "data" / "evaluation" / "questions.jsonl"
assert evaluation_path.is_file(), "Dataset d'évaluation introuvable."

# 2. Chargement des questions depuis le fichier JSONL
with evaluation_path.open("r", encoding="utf-8") as file:
    evaluation_questions = [json.loads(line) for line in file if line.strip()]

# 3. Vérif du nombre total de questions
assert len(evaluation_questions) == 50

# 4. Examen de la répartition réelle des ensembles
split_counts = Counter(str(item["split"]).casefold() for item in evaluation_questions)

print("=== Dataset d'évaluation ===")
print("Questions totales :", len(evaluation_questions))
print("Répartition :", dict(split_counts))

# 5. Séparation des questions de dev et de test
dev_questions = [item for item in evaluation_questions if str(item["split"]).casefold() in {"dev", "development"}]
test_questions = [item for item in evaluation_questions if str(item["split"]).casefold() == "test"]

# 6. Vérif de la séparation
assert len(dev_questions) == 10
assert len(test_questions) == 40
assert len(dev_questions) + len(test_questions) == len(evaluation_questions)

# 7. Vérif de la présence des questions
assert all(isinstance(item["question"], str) and item["question"].strip() for item in dev_questions)

# 8. Affichage des questions de développement seulement
print("\n=== Questions de développement ===")

for index, item in enumerate(dev_questions, start=1):
    question_id = item.get("question_id", item.get("id", f"dev_{index:02d}"))

    print(f"\n{index}. ID : {question_id}")
    print("Domaine :", item.get("topic", item.get("domain", "Non précisé")))
    print("Question :", item["question"])

# 9. Examen des champs disponibles pour l'annotation
print(("\n=== Structure documentaire ==="))

print("Champs d'une question :", sorted(dev_questions[0].keys()))
print("Champs d'un chunk :", sorted(chunks[0].keys()))

# 10. Confirmation de la préparation.
print("\n[OK] Dataset d'évaluation chargé.")
print("[OK] 10 questions de développement identifiées.")
print("[OK] 40 questions de test réservées.")
print("[OK] Aucune donnée modifiée.")

### *5.2. Construction des jugements de pertinence*

**Objectif :** identifier les passages documentaires
qui répondent réellement aux questions
de développement.

**Méthodologie :**

1. Définir une échelle de pertinence de 0 à 2.
2. Examiner les questions, les réponses de référence
   et leurs points clés attendus.
3. Parcourir les 55 chunks documentaires.
4. Lire les passages potentiellement pertinents.
5. Attribuer des jugements indépendamment des
   scores de similarité du moteur FAISS.
6. Conserver les identifiants des passages évalués.
7. Vérifier la couverture des annotations.

**Échelle de pertinence :**

- 0 : non pertinent.
- 1 : partiellement pertinent.
- 2 : directement pertinent.
- Non évalué : jugement non encore attribué.

**Première expérience :**

Étudier la question `CV001` concernant la suppression
de petits objets blancs dans une image binaire.

**Résultats attendus :**

- Question et réponse de référence identifiées.
- Liste des 55 passages disponibles.
- Contenu des passages morphologiques examiné.
- Premiers candidats à l'annotation identifiés.

**Points de vigilance :**

- Ne pas utiliser les scores FAISS pour attribuer les notes.
- Ne pas assimiler un domaine identique à une pertinence certaine.
- Ne pas assimiler un passage non lu à une note de 0.
- Ne pas consulter les réponses du jeu de test réservé.

In [ ]:
# 1. Association des questions de développement à leurs IDs.
development_by_id = {item["id"]: item for item in dev_questions}

assert len(development_by_id) == 10
assert len(chunks) == 55

# 2. Sélection de la première question de développement
question_id = "CV001"
evaluation_item = development_by_id[question_id]

# 3. Affichage de la question et des éléments de référence.
print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

# 4. Construction d'un catalogue de tous les chunks.
print("\n=== Catalogue documentaire ===")

for chunk in sorted(chunks, key=lambda item: (item["topic"], item["chunk_id"])):
    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['topic']} | "
        f"{chunk['section']}"
    )

# 5. Sélection des passages du même domaine
domain_chunks = [chunk for chunk in chunks if chunk["topic"] == evaluation_item["topic"]]
assert len(domain_chunks) == 10

# 6. Affichage de leur contenu pour l'inspection humaine.
print("\n=== Passages à examiner en priorité ===")

for chunk in domain_chunks:
    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Section :", chunk["section"])

    print("\nContenu documentaire :")
    print(chunk["text"])

# 7. Vérif du périmètre de l'inspection.
print("\n=== Bilan ===")

print("Corpus total :", len(chunks))
print("Passages du domaine :", len(domain_chunks))
print("Passages des autres domaines à examiner ensuite :", len(chunks) - len(domain_chunks))

print("\n[OK] Première inspection documentaire préparée.")
print("[INFO] Aucun jugement de pertinence enregistré.")

In [ ]:
# 1. Domaines restants à examiner
remaining_topics = ["filtering", "segmentation", "edges", "industrial_inspection"]

# 2. Choix du domaine à examiner (à modifier pour les autres exécutions)
topic_to_review = "industrial_inspection"
assert topic_to_review in remaining_topics

# 3. Sélection des chunks du domaine
review_chunks = [chunk for chunk in chunks if chunk["topic"] == topic_to_review]

# 4. Vérif de la disponibilité des passages
assert len(review_chunks) > 0

# 5. Affichage des textes complets
print("=== Inspection documentaire : CV001 ===")
print("Domaine examiné :", topic_to_review)
print("Passages à examiner :", len(review_chunks))

for chunk in review_chunks:
    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

# 6. Rappel des règles d'annotation.
print("\n=== Règles d'annotation ===")
print("0 : non pertinent")
print("1 : partiellement pertinent")
print("2 : directement pertinent")
print("Non évalué : passage non encore examiné")

print("\n[OK] Passages affichés pour inspection.")
print("[INFO] Aucun jugement enregistré.")

#### Enregistrement des annotations de CV001

**Objectif :** enregistrer les jugements de pertinence après inspection exhaustive des 55 chunks.

**Règle retenue :**

- 2 : passage directement pertinent.
- 1 : passage apportant une information ciblée mais incomplète.
- 0 : passage non pertinent ou insuffisamment ciblé.

**Question annotée :** CV001.

**Passages directement pertinents :**
- `opencv_morphology_chunk_005`

**Passages partiellement pertinents :**
- `opencv_morphology_chunk_003`
- `opencv_morphology_chunk_004`

**Méthodologie :**

1. Initialiser les jugements des 55 passages examinés.
2. Renseigner explicitement les notes 1 et 2.
3. Vérifier la couverture et les identifiants.
4. Enregistrer les annotations dans un fichier distinct du dataset d'évaluation figé.
5. Vérifier le contenu sauvegardé.

**Résultat attendu :**

55 jugements pour CV001 : un à 2, deux à 1 et cinquante-deux à 0.

**Point de vigilance :**

L'annotation est une référence humaine de développement,
distincte des scores produits par FAISS.

In [ ]:
import json
from collections import Counter

# 1. Construction de l'encsmeble des ids documentaires
all_chunk_ids = {chunk["chunk_id"] for chunk in chunks}
assert len(all_chunk_ids) == 55

# 2. Initialisation des notes des 55 chunks examinés
cv001_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

# 3. Attribution des notes validées par inspection
cv001_labels.update({
    "opencv_morphology_chunk_003": 1,
    "opencv_morphology_chunk_004": 1,
    "opencv_morphology_chunk_005": 2
})

# 4. Vérif de la couverture et des valeurs
assert set(cv001_labels) == all_chunk_ids
assert set(cv001_labels.values()).issubset({0 ,1, 2})

label_counts = Counter(cv001_labels.values())
assert label_counts == {0: 52, 1: 2, 2: 1}

# 5. Définition du chemin du fichier d'annotations
qrels_path =  project_dir / "data" / "evaluation" / "retrieval_qrels_dev.json"

# 6. Chargement des annotations existantes, si présentes.
if qrels_path.is_file():
    with qrels_path.open("r", encoding="utf-8") as file:
        qrels_data = json.load(file)
else:
    qrels_data = {
        "schema_version": 1,
        "relevance_scale": {
            "0": "non_relevant",
            "1": "partially_relevant",
            "2": "directly_relevant"
        },
        "annotations": {}
    }


# 7. Ajout de CV001 sans écraser d'autres annotations.
existing_cv001 = qrels_data["annotations"].get("CV001")
if existing_cv001 is not None:
    assert existing_cv001 == cv001_labels, "CV001 possède déjà des annotations différentes."

qrels_data["annotations"]["CV001"] = cv001_labels

# 8. Préparation du contenu JSON
qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

# 9. Pour éviter une reécriture inutile
if (qrels_path.is_file() and qrels_path.read_text(encoding="utf-8") == qrels_content):
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations enregistrées.")

# 10. Bilan
print("\n=== Jugements de pertinence : CV001 ===")

print("Chunks annotés :", len(cv001_labels))
print("Note 0 :", label_counts[0])
print("Note 1 :", label_counts[1])
print("Note 2 :", label_counts[2])

print("\nFichier :", qrels_path.name)

print("\n[OK] Couverture documentaire complète.")
print("[OK] Identifiants et notes validés.")
print("[OK] Annotations sauvegardées et rechargées.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV002 — Fermeture morphologique

**Objectif :** identifier les passages permettant de répondre à la deuxième question de développement.

**Méthodologie :**

1. Examiner la réponse de référence et ses points clés.
2. Réutiliser notre connaissance du corpus OpenCV.
3. Comparer les opérations d'érosion, de dilatation, d'ouverture et de fermeture.
4. Identifier les passages directement ou partiellement pertinents.
5. Vérifier les autres documents du corpus.
6. Enregistrer les jugements après validation.

**Point de vigilance :**

La fermeture morphologique et l'ouverture morphologique sont deux opérations distinctes.

Un passage pertinent pour CV001 n'est donc pas nécessairement pertinent pour CV002.

In [ ]:
# 1. Sélection de la deuxième question.
question_id = "CV002"
evaluation_item = development_by_id[question_id]

# 2. Affichage la question et sa référence.
print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])


# 3. Sélection des passages morphologiques potentiellement utiles pour cette question.
candidate_ids = [
    "opencv_morphology_chunk_003",
    "opencv_morphology_chunk_004",
    "opencv_morphology_chunk_005",
    "opencv_morphology_chunk_006",
    "opencv_morphology_chunk_010"
]


# 4. Pour retrouver les chunks grâce à leurs identifiants.
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)


# 5. Affichage des passages candidats.
print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:

    chunk = chunks_by_id[chunk_id]
    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])


# 6. Vérif que l'ensemble du corpus reste disponible pour l'annotation.
assert len(chunks_by_id) == 55

print("\n=== Bilan ===")
print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV002 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV002

**Objectif :** ajouter les jugements de CV002 au fichier d'annotations de développement.

**Jugements retenus :**

- Note 2 : `opencv_morphology_chunk_006`.
- Note 1 : `opencv_morphology_chunk_004`.
- Note 0 : les 53 autres chunks.

**Méthodologie :**

1. Initialiser les 55 jugements de CV002.
2. Renseigner les passages pertinents.
3. Charger le fichier d'annotations existant.
4. Conserver les jugements de CV001.
5. Ajouter CV002 sans écrasement inutile.
6. Vérifier les annotations après rechargement.

**Résultat attendu :**

Deux questions de développement annotées, avec 55 jugements chacune.

**Point de vigilance :**

Les annotations doivent rester indépendantes des scores de similarité FAISS.

In [ ]:
import json
from collections import Counter

# 1. Initialisation des notes des 55 chunks examinés.
cv002_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

# 2. Attribution des jugements retenus.
cv002_labels.update({
    "opencv_morphology_chunk_004": 1,
    "opencv_morphology_chunk_006": 2
})

# 3. Vérifi de la couverture et les notes.
assert set(cv002_labels) == all_chunk_ids

cv002_counts = Counter(cv002_labels.values())
assert cv002_counts == {0: 53, 1: 1, 2: 1}

# 4. Rechargement des annotations déjà enregistrées.
with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)


# 5. Vérif de l'integrité de CV001.
assert qrels_data["annotations"]["CV001"] == cv001_labels

# 6. Pour éviter d'écraser une éventuelle annotation différente.
existing_cv002 = qrels_data["annotations"].get("CV002")

if existing_cv002 is not None:
    assert existing_cv002 == cv002_labels, "CV002 possède déjà des annotations différentes."

# 7. Ajout des nouveaux jugements.
qrels_data["annotations"]["CV002"] = cv002_labels

# 8. Préparation du fichier JSON.
qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

# 9. Sauvegarde uniquement si le contenu change.
if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV002 enregistrées.")


# 10. Recharge et vérif des deux questions.
with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels["annotations"]["CV001"] == cv001_labels
assert saved_qrels["annotations"]["CV002"] == cv002_labels


# 11. Affichage du bilan.
print("\n=== Jugements de pertinence : CV002 ===")

print("Chunks annotés :", len(cv002_labels))
print("Note 0 :", cv002_counts[0])
print("Note 1 :", cv002_counts[1])
print("Note 2 :", cv002_counts[2])

print("\n=== Bilan des annotations ===")
print("Questions annotées :", len(saved_qrels["annotations"]))
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 conservée.")
print("[OK] CV002 sauvegardée et vérifiée.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV003 — Seuillage adaptatif

**Objectif :** déterminer quels passages expliquent pourquoi le seuillage adaptatif peut être préférable au seuillage global sous éclairage non uniforme.

**Méthodologie :**

1. Examiner la réponse de référence de CV003.
2. Identifier les passages traitant du seuillage global.
3. Examiner les passages décrivant le seuillage adaptatif.
4. Comparer l'utilisation d'un seuil unique et le calcul de seuils locaux.
5. Repérer les passages directement ou partiellement pertinents.
6. Vérifier les autres chunks avant l'enregistrement.

**Critères de pertinence :**

- La différence entre seuillage global et adaptatif.
- Les effets d'un éclairage non uniforme.
- Le calcul d'un seuil à partir du voisinage local.

**Résultat attendu :**

Une première proposition d'annotation pour les passages candidats de CV003.

**Point de vigilance :**

Le seuillage adaptatif et la méthode d'Otsu ne sont pas équivalents.
Dans sa forme classique, Otsu calcule un seuil global automatiquement.

In [ ]:
# 1. Sélection de la troisième question.
question_id = "CV003"
evaluation_item = development_by_id[question_id]

# 2. Affichage des informations de référence.
print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

# 3. Identification des passages candidats.
# Nous examinons les deux types de seuillage et la méthode d'Otsu pour comparer leurs principes.
candidate_ids = [
    "opencv_thresholding_chunk_002",
    "opencv_thresholding_chunk_003",
    "opencv_thresholding_chunk_004",
    "opencv_thresholding_chunk_005",
    "opencv_thresholding_chunk_006"
]

# 4. Vérif de la présence de chaque passage.
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

# 5. Affichage de leur contenu documentaire.
print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])


# 6. Vérif du périmètre documentaire.
assert len(chunks_by_id) == 55
assert len(candidate_ids) == 5

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV003 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV003

**Objectif :** enregistrer les jugements de pertinence de CV003 après examen du corpus documentaire.

**Jugements retenus :**

- Note 2 : `opencv_thresholding_chunk_004`.
- Note 1 : `opencv_thresholding_chunk_002`, `opencv_thresholding_chunk_003` et `opencv_thresholding_chunk_005`.
- Note 0 : les 51 autres chunks.

**Méthodologie :**

1. Initialiser les jugements des 55 passages.
2. Renseigner les notes 1 et 2.
3. Vérifier la couverture documentaire.
4. Recharger les annotations précédentes.
5. Préserver les jugements de CV001 et CV002.
6. Enregistrer CV003 sans écrasement inutile.
7. Recharger et vérifier les trois annotations.

**Résultat attendu :**

Trois questions de développement annotées, avec 55 jugements chacune, soit 165 jugements.

**Point de vigilance :**

Le seuillage adaptatif utilise des seuils locaux, tandis que la méthode d'Otsu classique calcule un seuil global automatiquement.

In [ ]:
import json
from collections import Counter

# 1. Initialisation les notes des 55 chunks examinés.
cv003_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

# 2. Attribution des jugements de pertinence retenus.
cv003_labels.update({
    "opencv_thresholding_chunk_002": 1,
    "opencv_thresholding_chunk_003": 1,
    "opencv_thresholding_chunk_004": 2,
    "opencv_thresholding_chunk_005": 1
})

# 3. Vérif de la couverture documentaire.
assert set(cv003_labels) == all_chunk_ids
assert set(cv003_labels.values()).issubset({0, 1, 2})

cv003_counts = Counter(cv003_labels.values())
assert cv003_counts == {0: 51, 1: 3, 2: 1}

# 4. Recharge des annotations existantes.
with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

# 5. Vérif que CV001 et CV002 sont conservées.
assert qrels_data["annotations"]["CV001"] == cv001_labels
assert qrels_data["annotations"]["CV002"] == cv002_labels

# 6. Vérif d'une éventuelle annotation CV003 existante.
existing_cv003 = qrels_data["annotations"].get("CV003")
if existing_cv003 is not None:
    assert existing_cv003 == cv003_labels, "CV003 possède déjà des annotations différentes."

# 7. Ajout des jugements de CV003.
qrels_data["annotations"]["CV003"] = cv003_labels

# 8. Préparation du contenu JSON.
qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

# 9. Sauvegarde uniquement si le contenu change.
if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text( qrels_content, encoding="utf-8")
    print("[OK] Annotations CV003 enregistrées.")

# 10. Recharge des annotations sauvegardées.
with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

# 11. Vérif des trois questions.
assert saved_qrels["annotations"]["CV001"] == cv001_labels
assert saved_qrels["annotations"]["CV002"] == cv002_labels
assert saved_qrels["annotations"]["CV003"] == cv003_labels

assert saved_qrels == qrels_data

# 12. Vérif du bilan global.
annotated_questions = len(saved_qrels["annotations"])
assert annotated_questions == 3

total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert total_judgments == 165

# 13. Affichage des résultats.
print("\n=== Jugements de pertinence : CV003 ===")

print("Chunks annotés :", len(cv003_labels))
print("Note 0 :", cv003_counts[0])
print("Note 1 :", cv003_counts[1])
print("Note 2 :", cv003_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 et CV002 conservées.")
print("[OK] CV003 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV004 — Paramètre ksize du filtre médian

**Objectif :** identifier les passages documentaires expliquant le rôle et les contraintes du paramètre `ksize` dans `cv2.medianBlur()`.

**Méthodologie :**

1. Examiner la réponse de référence de CV004.
2. Étudier le fonctionnement du filtre médian.
3. Identifier le rôle du voisinage de filtrage.
4. Examiner les contraintes imposées à `ksize`.
5. Comparer les passages traitant d'autres filtres.
6. Proposer des jugements de pertinence.
7. Vérifier la couverture du corpus avant sauvegarde.

**Critères de pertinence :**

- Rôle du paramètre `ksize`.
- Taille du voisinage utilisé pour la médiane.
- Contrainte sur la valeur de `ksize`.

**Résultat attendu :**

Une première proposition de jugements pour les cinq passages candidats.

**Point de vigilance :**

Un passage décrivant un noyau de filtrage n'est pas nécessairement pertinent pour les contraintes propres à `cv2.medianBlur()`.

In [ ]:
# 1. Sélection de la quatrième question.
question_id = "CV004"
evaluation_item = development_by_id[question_id]

# 2. Affichage de la question et ses éléments de référence.
print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

# 3. Sélection des passages candidats.
# Nous comparons le filtre médian aux autres filtres et aux éléments structurants morphologiques.
candidate_ids = [
    "opencv_filtering_chunk_004",
    "opencv_filtering_chunk_005",
    "opencv_filtering_chunk_006",
    "opencv_filtering_chunk_007",
    "opencv_morphology_chunk_010"
]

# 4. Vérif que les cinq passages existent.
assert len(candidate_ids) == 5
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

# 5. Affichage du contenu des passages candidats.
print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:

    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])


# 6. Confirmation du périmètre documentaire.
assert len(chunks_by_id) == 55

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV004 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV004

**Objectif :** enregistrer les jugements de pertinence de CV004 après inspection du corpus documentaire.

**Jugements retenus :**

- Note 2 : `opencv_filtering_chunk_006`.
- Note 1 : aucun passage.
- Note 0 : les 54 autres chunks.

**Méthodologie :**

1. Initialiser les jugements des 55 chunks.
2. Identifier le passage directement pertinent.
3. Vérifier les identifiants et les notes.
4. Recharger les annotations existantes.
5. Préserver les jugements de CV001 à CV003.
6. Ajouter CV004 sans écrasement inutile.
7. Recharger et vérifier les quatre annotations.

**Résultat attendu :**

Quatre questions de développement annotées, soit 220 jugements de pertinence.

**Point de vigilance :**

Le paramètre `ksize` existe dans plusieurs fonctions OpenCV, mais ses contraintes doivent être vérifiées pour la fonction spécifiquement demandée.

In [ ]:
import json
from collections import Counter

cv004_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

cv004_labels.update({
    "opencv_filtering_chunk_006": 2
})


assert set(cv004_labels) == all_chunk_ids
assert set(cv004_labels.values()).issubset({0, 1, 2})

cv004_counts = Counter(cv004_labels.values())
assert cv004_counts == {0: 54, 2: 1}

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

assert qrels_data["annotations"]["CV001"] == cv001_labels
assert qrels_data["annotations"]["CV002"] == cv002_labels
assert qrels_data["annotations"]["CV003"] == cv003_labels

existing_cv004 = qrels_data["annotations"].get("CV004")
if existing_cv004 is not None:
    assert existing_cv004 == cv004_labels, "CV004 possède déjà des annotations différentes."


qrels_data["annotations"]["CV004"] = cv004_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV004 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels["annotations"]["CV001"] == cv001_labels
assert saved_qrels["annotations"]["CV002"] == cv002_labels
assert saved_qrels["annotations"]["CV003"] == cv003_labels
assert saved_qrels["annotations"]["CV004"] == cv004_labels

assert saved_qrels == qrels_data

annotated_questions = len(saved_qrels["annotations"])
assert annotated_questions == 4

total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert total_judgments == 220

print("\n=== Jugements de pertinence : CV004 ===")

print("Chunks annotés :", len(cv004_labels))
print("Note 0 :", cv004_counts[0])
print("Note 1 :", cv004_counts[1])
print("Note 2 :", cv004_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001, CV002 et CV003 conservées.")
print("[OK] CV004 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV005 — Filtrage bilatéral et préservation des contours

**Objectif :** identifier les passages qui expliquent pourquoi le filtre bilatéral préserve généralement mieux les contours qu'un filtre gaussien classique.

**Méthodologie :**

1. Examiner la réponse de référence et ses points clés.
2. Étudier les principes du filtrage gaussien.
3. Examiner le fonctionnement du filtre bilatéral.
4. Distinguer les pondérations spatiale et photométrique.
5. Identifier les passages directement ou partiellement pertinents.
6. Vérifier les autres documents du corpus.
7. Enregistrer les jugements après validation.

**Critères de pertinence :**

- Prise en compte de la proximité spatiale.
- Prise en compte des différences d'intensité.
- Réduction du mélange entre pixels de part et d'autre d'un contour.
- Comparaison avec le filtrage gaussien classique.

**Résultat attendu :**

Une proposition d'annotation documentée pour les passages candidats de CV005.

**Point de vigilance :**

Un passage évoquant simplement le débruitage ou la préservation des contours ne suffit pas nécessairement à expliquer le mécanisme bilatéral.

In [ ]:
question_id = "CV005"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_filtering_chunk_003",
    "opencv_filtering_chunk_004",
    "opencv_filtering_chunk_005",
    "opencv_filtering_chunk_006",
    "opencv_filtering_chunk_007"
]

assert len(candidate_ids) == 5
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:

    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

assert len(chunks_by_id) == 55

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV005 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV005

**Objectif :** enregistrer les jugements de pertinence de CV005 après vérification des 55 chunks OpenCV.

**Jugements retenus :**

- Note 2 : `opencv_filtering_chunk_007`.
- Note 1 : `opencv_filtering_chunk_003`.
- Note 0 : les 53 autres chunks.

**Méthodologie :**

1. Initialiser les 55 jugements de CV005.
2. Attribuer les notes 1 et 2.
3. Vérifier les identifiants et la couverture du corpus.
4. Charger les annotations précédentes.
5. Préserver les jugements de CV001 à CV004.
6. Ajouter CV005 sans écrasement inutile.
7. Recharger et vérifier les cinq annotations.

**Résultat attendu :**

Cinq questions de développement annotées, soit 275 jugements de pertinence.

**Point de vigilance :**

La pertinence dépend de l'explication réellement contenue dans le passage, et non uniquement du domaine ou du nom du filtre mentionné.

In [ ]:
import json
from collections import Counter

cv005_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}
cv005_labels.update({
    "opencv_filtering_chunk_003": 1,
    "opencv_filtering_chunk_007": 2
})

assert set(cv005_labels) == all_chunk_ids
assert set(cv005_labels.values()).issubset({0, 1, 2})

cv005_counts = Counter(cv005_labels.values())
assert cv005_counts == {0: 53, 1: 1, 2: 1}

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels
}

for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] == (expected_labels)


existing_cv005 = qrels_data["annotations"].get("CV005")
if existing_cv005 is not None:
    assert existing_cv005 == cv005_labels, "CV005 possède déjà des annotations différentes."

qrels_data["annotations"]["CV005"] = cv005_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent= 2) + "\n"

if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")

    print("[OK] Annotations CV005 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)   

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels

assert saved_qrels["annotations"]["CV005"] == cv005_labels

annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 5
assert total_judgments == 275

print("\n=== Jugements de pertinence : CV005 ===")

print("Chunks annotés :", len(cv005_labels))
print("Note 0 :", cv005_counts[0])
print("Note 1 :", cv005_counts[1])
print("Note 2 :", cv005_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV004 conservées.")
print("[OK] CV005 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Dataset original inchangé.")


#### Annotation de CV006 — Méthode d'Otsu

**Objectif :** identifier les passages expliquant la différence entre un seuillage global manuel et la méthode d'Otsu.

**Méthodologie :**

1. Examiner la réponse de référence de CV006.
2. Étudier le seuillage global manuel.
3. Examiner la sélection automatique du seuil par la méthode d'Otsu.
4. Comprendre le rôle de l'histogramme.
5. Examiner le critère de variance intra-classe.
6. Distinguer seuillage automatique et seuillage local.
7. Proposer les jugements de pertinence.
8. Vérifier le reste du corpus avant sauvegarde.

**Critères de pertinence :**

- Seuil global fixé manuellement.
- Détermination automatique du seuil par Otsu.
- Exploitation de l'histogramme.
- Critère d'optimisation utilisé par Otsu.

**Résultat attendu :**

Une première proposition d'annotation pour cinq passages documentaires.

**Point de vigilance :**

La méthode d'Otsu classique utilise un seuil global, même si ce seuil est déterminé automatiquement. Elle ne doit pas être confondue avec le seuillage adaptatif local.

In [ ]:
question_id = "CV006"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_thresholding_chunk_002",
    "opencv_thresholding_chunk_004",
    "opencv_thresholding_chunk_006",
    "opencv_thresholding_chunk_007",
    "opencv_thresholding_chunk_008"
]

assert len(candidate_ids) == 5
assert len(set(candidate_ids)) == 5
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

assert len(chunks_by_id) == 55

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV006 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV006

**Objectif :** enregistrer les jugements de pertinence pour la comparaison entre Otsu et le seuillage global manuel.

**Jugements retenus :**

- Note 2 : `opencv_thresholding_chunk_006`.
- Note 1 : `opencv_thresholding_chunk_002`,
  `opencv_thresholding_chunk_003`,
  `opencv_thresholding_chunk_007`,
  `opencv_thresholding_chunk_008`,
  `opencv_thresholding_chunk_009`.
- Note 0 : les 49 autres chunks.

**Méthodologie :**

1. Initialiser les jugements des 55 chunks.
2. Renseigner les notes 1 et 2.
3. Vérifier les identifiants et les notes.
4. Recharger les cinq annotations précédentes.
5. Ajouter CV006 sans modifier CV001 à CV005.
6. Sauvegarder sans écrasement inutile.
7. Recharger et vérifier les six annotations.

**Résultat attendu :**

Six questions de développement annotées, soit 330 jugements de pertinence.

**Point de vigilance :**

Une sélection automatique du seuil ne signifie pas nécessairement un seuillage local.

In [ ]:
import json
from collections import Counter

cv006_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}
cv006_labels.update({
    "opencv_thresholding_chunk_002": 1,
    "opencv_thresholding_chunk_003": 1,
    "opencv_thresholding_chunk_006": 2,
    "opencv_thresholding_chunk_007": 1,
    "opencv_thresholding_chunk_008": 1,
    "opencv_thresholding_chunk_009": 1
})

assert set(cv006_labels) == all_chunk_ids
assert set(cv006_labels.values()).issubset({0, 1, 2})

cv006_counts = Counter(cv006_labels.values())
assert cv006_counts == {0: 49, 1: 5, 2: 1}

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels,
    "CV005": cv005_labels
}

for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] == expected_labels

existing_cv006 = qrels_data["annotations"].get("CV006")

if existing_cv006 is not None:
    assert existing_cv006 == cv006_labels, "CV006 possède déjà des annotations différentes."

qrels_data["annotations"]["CV006"] = cv006_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"


if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV006 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels
    
assert saved_qrels["annotations"]["CV006"] == cv006_labels

annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 6
assert total_judgments == 330

print("\n=== Jugements de pertinence : CV006 ===")

print("Chunks annotés :", len(cv006_labels))
print("Note 0 :", cv006_counts[0])
print("Note 1 :", cv006_counts[1])
print("Note 2 :", cv006_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV005 conservées.")
print("[OK] CV006 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV007 — Hystérésis de Canny

**Objectif :** identifier les passages expliquant le rôle des deux seuils dans l'étape d'hystérésis de l'algorithme de Canny.

**Méthodologie :**

1. Examiner la réponse de référence de CV007.
2. Identifier le rôle du seuil inférieur.
3. Identifier le rôle du seuil supérieur.
4. Étudier la conservation des contours faibles reliés à des contours forts.
5. Distinguer l'hystérésis des autres étapes de l'algorithme de Canny.
6. Proposer les jugements de pertinence.
7. Vérifier le reste du corpus avant sauvegarde.

**Critères de pertinence :**

- Identification des contours forts.
- Rejet des pixels sous le seuil inférieur.
- Conservation conditionnelle des contours faibles.
- Rôle de la connectivité dans l'hystérésis.

**Résultat attendu :**

Une première proposition d'annotation pour cinq passages documentaires.

**Point de vigilance :**

Les deux seuils ne servent pas à segmenter directement les intensités de l'image originale.

Ils interviennent sur les réponses de gradient utilisées par l'algorithme de détection de contours.

In [ ]:
question_id = "CV007"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_canny_chunk_001",
    "opencv_canny_chunk_002",
    "opencv_canny_chunk_003",
    "opencv_canny_chunk_004",
    "opencv_canny_chunk_005"
]

assert len(candidate_ids) == 5
assert len(set(candidate_ids)) == 5
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

assert len(chunks_by_id) == 55

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV007 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV007

**Objectif :** enregistrer les jugements de pertinence
pour le rôle des deux seuils de l'hystérésis de Canny.

**Jugements retenus :**

- Note 2 : `opencv_canny_chunk_003`.
- Note 1 : aucun passage.
- Note 0 : les 54 autres chunks.

**Méthodologie :**

1. Initialiser les jugements des 55 passages.
2. Attribuer la note du passage directement pertinent.
3. Vérifier les identifiants et les notes.
4. Recharger les annotations précédentes.
5. Préserver CV001 à CV006.
6. Ajouter CV007 sans écrasement inutile.
7. Recharger et vérifier les sept annotations.

**Résultat attendu :**

Sept questions de développement annotées, soit 385 jugements de pertinence.

**Point de vigilance :**

La simple présence des paramètres `minVal` et `maxVal` ne suffit pas à expliquer le mécanisme de l'hystérésis et la connectivité des contours.

In [ ]:
import json
from collections import Counter

cv007_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}
cv007_labels.update({
    "opencv_canny_chunk_003": 2
})

assert set(cv007_labels) == all_chunk_ids
assert set(cv007_labels.values()).issubset({0, 1, 2})

cv007_counts = Counter(cv007_labels.values())
assert cv007_counts == {0: 54, 2: 1}

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels,
    "CV005": cv005_labels,
    "CV006": cv006_labels
}

for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] ==  expected_labels


existing_cv007 = qrels_data["annotations"].get("CV007")

if existing_cv007 is not None:
    assert existing_cv007 == cv007_labels, "CV007 possède déjà des annotations différentes."

qrels_data["annotations"]["CV007"] = cv007_labels


qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV007 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels

assert saved_qrels["annotations"]["CV007"] == cv007_labels

annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 7
assert total_judgments == 385


print("\n=== Jugements de pertinence : CV007 ===")

print("Chunks annotés :", len(cv007_labels))
print("Note 0 :", cv007_counts[0])
print("Note 1 :", cv007_counts[1])
print("Note 2 :", cv007_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV006 conservées.")
print("[OK] CV007 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Dataset original inchangé.")


#### Annotation de CV008 — Image binaire et findContours

**Objectif :** déterminer si le corpus contient des passages expliquant pourquoi une image binaire est généralement utilisée avant `cv2.findContours()`.

**Méthodologie :**

1. Examiner la réponse de référence de CV008.
2. Identifier les passages utilisant `findContours`.
3. Examiner les explications sur la binarisation.
4. Distinguer une image binaire d'une image en niveaux de gris.
5. Étudier les éventuels liens entre binarisation et extraction de contours.
6. Proposer des jugements de pertinence.
7. Vérifier les autres chunks avant sauvegarde.

**Critères de pertinence :**

- Distinction entre arrière-plan et premier plan.
- Utilisation d'une image binaire.
- Extraction des frontières des objets.
- Rôle de cette préparation pour `findContours`.

**Résultat attendu :**

Une proposition d'annotation pour six passages documentaires potentiellement utiles.

**Point de vigilance :**

Il est possible qu'aucun chunk ne fournisse une réponse directement pertinente.

Nous devons documenter cette éventuelle absence de couverture plutôt que d'inventer un passage de référence.

In [ ]:
question_id = "CV008"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_contours_chunk_002",
    "opencv_contours_chunk_003",
    "opencv_contours_chunk_004",
    "opencv_thresholding_chunk_002",
    "opencv_thresholding_chunk_003",
    "opencv_canny_chunk_003"
]

assert len(candidate_ids) == 6
assert len(set(candidate_ids)) == 6
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

assert len(chunks_by_id) == 55

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV008 affichés.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV008

**Objectif :** enregistrer les jugements de pertinence pour la préparation d'une image binaire avant l'utilisation de `cv2.findContours()`.

**Jugements retenus :**

- Note 2 : aucun passage.
- Note 1 : `opencv_thresholding_chunk_002` et `opencv_contours_chunk_010`.
- Note 0 : les 53 autres chunks.

**Méthodologie :**

1. Initialiser les 55 jugements documentaires.
2. Attribuer les notes aux passages partiellement pertinents.
3. Vérifier l'absence de passage directement pertinent.
4. Recharger les annotations précédentes.
5. Préserver CV001 à CV007.
6. Ajouter CV008 sans écrasement inutile.
7. Recharger et vérifier les huit annotations.

**Résultat attendu :**

Huit questions de développement annotées, soit 440 jugements de pertinence.

**Limite documentaire identifiée :**

Aucun passage du corpus ne justifie complètement le rôle de la binarisation avant `findContours()`.

**Point de vigilance :**

L'absence de passage de note 2 doit être conservée dans les annotations et prise en compte lors du calcul des métriques de retrieval.

In [ ]:
import json
from collections import Counter

cv008_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

cv008_labels.update({
    "opencv_thresholding_chunk_002": 1,
    "opencv_contours_chunk_010": 1
})

assert set(cv008_labels) == all_chunk_ids
assert set(cv008_labels.values()).issubset({0, 1, 2})

cv008_counts = Counter(cv008_labels.values())
assert cv008_counts == {0: 53, 1: 2}
assert cv008_counts[2] == 0

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels,
    "CV005": cv005_labels,
    "CV006": cv006_labels,
    "CV007": cv007_labels
}

for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] == expected_labels

existing_cv008 = qrels_data["annotations"].get("CV008")
if existing_cv008 is not None:
    assert existing_cv008 == cv008_labels, "CV008 possède déjà des annotations différentes."

qrels_data["annotations"]["CV008"] = cv008_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV008 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels

assert saved_qrels["annotations"]["CV008"] == cv008_labels

annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 8
assert total_judgments == 440

print("\n=== Jugements de pertinence : CV008 ===")

print("Chunks annotés :", len(cv008_labels))
print("Note 0 :", cv008_counts[0])
print("Note 1 :", cv008_counts[1])
print("Note 2 :", cv008_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV007 conservées.")
print("[OK] CV008 sauvegardée et vérifiée.")
print("[OK] Couverture documentaire complète.")
print("[OK] Absence de passage directement pertinent documentée.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV009 — Aire d'un contour

**Objectif :** déterminer si le corpus permet d'expliquer comment mesurer l'aire d'un défaut segmenté représenté par un contour dans OpenCV.

**Méthodologie :**

1. Examiner la réponse de référence de CV009.
2. Identifier la fonction OpenCV attendue.
3. Examiner les passages consacrés aux contours.
4. Distinguer l'extraction d'un contour de la mesure de son aire.
5. Examiner les autres méthodes de localisation et de segmentation présentes dans le corpus.
6. Rechercher les mentions explicites du calcul d'aire dans l'ensemble des 55 chunks.
7. Proposer les jugements de pertinence.
8. Vérifier la couverture avant sauvegarde.

**Critères de pertinence :**

- Identification d'un contour représentant un défaut.
- Calcul de l'aire délimitée par ce contour.
- Utilisation de la fonction OpenCV appropriée.
- Interprétation de la surface obtenue.

**Résultat attendu :**

Une proposition d'annotation fondée sur les passages réellement disponibles.

**Point de vigilance :**

Un passage expliquant `cv.findContours()` ne décrit pas nécessairement `cv.contourArea()`.

L'absence éventuelle de réponse documentaire doit être conservée dans le protocole.

In [ ]:
question_id = "CV009"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_contours_chunk_003",
    "opencv_contours_chunk_004",
    "opencv_contours_chunk_009",
    "opencv_contours_chunk_010",
    "opencv_thresholding_chunk_002",
    "opencv_template_matching_chunk_002"
]

assert len(candidate_ids) == 6
assert len(set(candidate_ids)) == 6
assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)

print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])

assert len(chunks_by_id) == 55

search_terms = [
    "contourarea",
    "contour area",
    "area of a contour"
]

matching_chunks = [chunk for chunk in chunks if any(term in chunk["text"].casefold() for term in search_terms)]

print("\n=== Recherche de mentions explicites ===")
print("Chunks correspondants :", len(matching_chunks))

for chunk in matching_chunks:
    print("-", chunk["chunk_id"], "|", chunk["section"])

print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV009 affichés.")
print("[OK] Recherche terminologique effectuée.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV009

**Objectif :** documenter l'absence de passage pertinent pour le calcul de l'aire d'un défaut représenté par un contour.

**Jugements retenus :**

- Note 2 : aucun passage.
- Note 1 : aucun passage.
- Note 0 : les 55 chunks du corpus.

**Méthodologie :**

1. Initialiser les 55 jugements de CV009.
2. Vérifier l'absence de passage pertinent.
3. Recharger les annotations précédentes.
4. Préserver les jugements de CV001 à CV008.
5. Enregistrer CV009 sans écrasement inutile.
6. Recharger et vérifier les neuf annotations.

**Résultat attendu :**

Neuf questions de développement annotées, soit 495 jugements de pertinence.

**Limite documentaire identifiée :**

Le corpus ne contient aucun passage expliquant comment calculer l'aire d'un contour avec `cv2.contourArea()`.

**Point de vigilance :**

Une question sans passage pertinent doit rester identifiée dans le jeu d'annotations.

Le calcul de Recall@k et de MRR devra appliquer une convention explicite à ces questions.

In [ ]:
import json
from collections import Counter

cv009_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}

assert set(cv009_labels) == all_chunk_ids
assert set(cv009_labels.values()) == {0}
cv009_counts = Counter(cv009_labels.values())
assert cv009_counts == {0: 55}
assert cv009_counts[1] == 0
assert cv009_counts[2] == 0

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels,
    "CV005": cv005_labels,
    "CV006": cv006_labels,
    "CV007": cv007_labels,
    "CV008": cv008_labels
}

for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] == expected_labels

existing_cv009 = qrels_data["annotations"].get("CV009")
if existing_cv009 is not None:
    assert existing_cv009 == cv009_labels, "CV009 possède déjà des annotations différentes."

qrels_data["annotations"]["CV009"] = cv009_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"

if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV009 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels

assert saved_qrels["annotations"]["CV009"] == cv009_labels

annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 9
assert total_judgments == 495

print("\n=== Jugements de pertinence : CV009 ===")

print("Chunks annotés :", len(cv009_labels))
print("Note 0 :", cv009_counts[0])
print("Note 1 :", cv009_counts[1])
print("Note 2 :", cv009_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV008 conservées.")
print("[OK] CV009 sauvegardée et vérifiée.")
print("[OK] Absence de passage pertinent documentée.")
print("[OK] Couverture des jugements complète.")
print("[OK] Dataset original inchangé.")

#### Annotation de CV010 — Rectangle englobant d'un défaut

**Objectif :** déterminer si le corpus explique comment calculer un rectangle englobant aligné sur les axes à partir d'un contour OpenCV.

**Méthodologie :**

1. Examiner la réponse de référence de CV010.
2. Identifier la fonction OpenCV attendue.
3. Étudier les passages consacrés aux contours.
4. Examiner les exemples de rectangles présents dans les documents de template matching.
5. Distinguer le calcul d'un rectangle à partir d'un contour et le dessin d'un rectangle.
6. Rechercher les mentions explicites dans le corpus.
7. Proposer les jugements de pertinence.
8. Vérifier les autres chunks avant sauvegarde.

**Critères de pertinence :**

- Calcul du rectangle englobant d'un contour.
- Coordonnées x et y du coin supérieur gauche.
- Largeur w et hauteur h.
- Rectangle aligné sur les axes de l'image.
- Localisation du défaut détecté.

**Résultat attendu :**

Une proposition d'annotation pour six passages documentaires, accompagnée d'une recherche terminologique sur les 55 chunks.

**Point de vigilance :**

Un rectangle dessiné après une recherche par gabarit ne constitue pas nécessairement une explication de `cv.boundingRect(contour)`.

Une absence de couverture documentaire doit être conservée dans nos jugements.

In [ ]:
question_id = "CV010"
evaluation_item = development_by_id[question_id]

print("=== Question à annoter ===")

print("ID :", evaluation_item["id"])
print("Domaine :", evaluation_item["topic"])
print("Question :", evaluation_item["question"])

print("\nRéponse de référence :")
print(evaluation_item["reference_answer"])

print("\nPoints clés attendus :")
print(evaluation_item["key_points"])

candidate_ids = [
    "opencv_contours_chunk_003",
    "opencv_contours_chunk_004",
    "opencv_contours_chunk_009",
    "opencv_template_matching_chunk_002",
    "opencv_template_matching_chunk_004",
    "opencv_template_matching_chunk_006"
]

assert len(candidate_ids) == 6
assert len(set(candidate_ids)) == 6

assert all(chunk_id in chunks_by_id for chunk_id in candidate_ids)


print("\n=== Passages candidats ===")

for chunk_id in candidate_ids:
    chunk = chunks_by_id[chunk_id]

    print("\n" + "=" * 60)
    print("ID :", chunk["chunk_id"])
    print("Domaine :", chunk["topic"])
    print("Section :", chunk["section"])

    print("\nContenu :")
    print(chunk["text"])


search_terms = [
    "boundingrect",
    "bounding rect",
    "bounding rectangle",
    "bounding box",
    "cv.rectangle"
]

matching_chunks = [chunk for chunk in chunks if any(term in chunk["text"].casefold() for term in search_terms)]

print("\n=== Recherche de mentions explicites ===")

print("Chunks correspondants :", len(matching_chunks))

for chunk in matching_chunks:
    matched_terms = [term for term in search_terms if term in chunk["text"].casefold()]
    print(chunk["chunk_id"], "|", chunk["section"], "| Termes :", matched_terms)

assert len(chunks_by_id) == 55
assert len(chunks) == 55


print("\n=== Bilan ===")

print("Passages candidats :", len(candidate_ids))
print("Corpus total :", len(chunks_by_id))

print("\n[OK] Passages candidats de CV010 affichés.")
print("[OK] Recherche terminologique effectuée.")
print("[INFO] Aucun nouveau jugement enregistré.")

#### Enregistrement des annotations de CV010

**Objectif :** documenter l'absence de passage pertinent pour le calcul d'un rectangle englobant à partir d'un contour avec `cv.boundingRect()`.

**Jugements retenus :**

- Note 2 : aucun passage.
- Note 1 : aucun passage.
- Note 0 : les 55 chunks.

**Méthodologie :**

1. Initialiser les 55 jugements de CV010.
2. Vérifier que chaque chunk reçoit la note 0.
3. Recharger les annotations précédentes.
4. Préserver les jugements de CV001 à CV009.
5. Ajouter CV010 sans écrasement inutile.
6. Recharger et vérifier les dix annotations.
7. Contrôler la couverture complète du jeu de développement.

**Résultat attendu :**

Dix questions de développement annotées, soit 550 jugements de pertinence.

**Limite documentaire identifiée :**

Le corpus ne contient pas d'explication du calcul d'un rectangle englobant à partir d'un contour avec `cv.boundingRect()`.

**Point de vigilance :**

Les exemples de `cv.rectangle()` dans le template matching dessinent des rectangles, mais ne décrivent pas leur calcul à partir de contours segmentés.

Les questions sans passage pertinent devront être traitées explicitement dans l'évaluation.

In [ ]:
import json
from collections import Counter

cv010_labels = {chunk_id: 0 for chunk_id in sorted(all_chunk_ids)}


assert set(cv010_labels) == all_chunk_ids
cv010_counts = Counter(cv010_labels.values())
assert cv010_counts == {0: 55}
assert cv010_counts[1] == 0
assert cv010_counts[2] == 0

with qrels_path.open("r", encoding="utf-8") as file:
    qrels_data = json.load(file)

previous_labels = {
    "CV001": cv001_labels,
    "CV002": cv002_labels,
    "CV003": cv003_labels,
    "CV004": cv004_labels,
    "CV005": cv005_labels,
    "CV006": cv006_labels,
    "CV007": cv007_labels,
    "CV008": cv008_labels,
    "CV009": cv009_labels
}


for previous_id, expected_labels in previous_labels.items():
    assert qrels_data["annotations"][previous_id] == expected_labels

existing_cv010 = qrels_data["annotations"].get("CV010")
if existing_cv010 is not None:
    assert existing_cv010 == cv010_labels, "CV010 possède déjà des annotations différentes."

qrels_data["annotations"]["CV010"] = cv010_labels

qrels_content = json.dumps(qrels_data, ensure_ascii=False, sort_keys=True, indent=2) + "\n"


if qrels_path.read_text(encoding="utf-8") == qrels_content:
    print("[INFO] Annotations déjà enregistrées et à jour.")
else:
    qrels_path.write_text(qrels_content, encoding="utf-8")
    print("[OK] Annotations CV010 enregistrées.")

with qrels_path.open("r", encoding="utf-8") as file:
    saved_qrels = json.load(file)

assert saved_qrels == qrels_data

for previous_id, expected_labels in previous_labels.items():
    assert saved_qrels["annotations"][previous_id] == expected_labels

assert saved_qrels["annotations"]["CV010"] == cv010_labels


expected_question_ids = {item["id"] for item in dev_questions}
assert set(saved_qrels["annotations"]) == expected_question_ids
assert len(expected_question_ids) == 10

for question_id, labels in saved_qrels["annotations"].items():
    assert set(labels) == all_chunk_ids
    assert set(labels.values()).issubset({0, 1, 2})


annotated_questions = len(saved_qrels["annotations"])
total_judgments = sum(len(labels) for labels in saved_qrels["annotations"].values())
assert annotated_questions == 10
assert total_judgments == 550


print("\n=== Jugements de pertinence : CV010 ===")

print("Chunks annotés :", len(cv010_labels))
print("Note 0 :", cv010_counts[0])
print("Note 1 :", cv010_counts[1])
print("Note 2 :", cv010_counts[2])

print("\n=== Bilan des annotations ===")

print("Questions annotées :", annotated_questions)
print("Jugements enregistrés :", total_judgments)
print("Fichier :", qrels_path.name)

print("\n[OK] CV001 à CV009 conservées.")
print("[OK] CV010 sauvegardée et vérifiée.")
print("[OK] 10 questions de développement annotées.")
print("[OK] 550 jugements documentaires vérifiés.")
print("[OK] Dataset original inchangé.")

### *5.3. Calcul des métriques de pertinence du retrieval*

**Objectif :** mesurer la qualité du classement documentaire produit par FAISS sur les dix questions de développement annotées.

**Méthodologie :**

1. Vérifier la couverture du jeu d'annotations.
2. Définir les passages pertinents à partir des notes.
3. Mesurer Precision@k et Recall@k.
4. Calculer le rang réciproque moyen (MRR).
5. Comparer les résultats pour différentes valeurs de k.
6. Distinguer performance du retriever et couverture documentaire.
7. Réaliser une analyse principale et une analyse stricte.

**Conventions :**

- Évaluation principale : notes 1 et 2 pertinentes.
- Évaluation stricte : seule la note 2 est pertinente.
- Les questions sans passage pertinent sont exclues des moyennes conditionnelles de Recall@k et MRR.
- La couverture du corpus est toujours indiquée.
- Une Precision@k sur toutes les questions est rapportée séparément.

**Résultats attendus :**

- Métriques reproductibles sur le développement.
- Comparaison des valeurs de k.
- Identification des erreurs de classement.
- Analyse des limites du corpus.

**Point de vigilance :**

Aucun paramètre ne sera ajusté à partir des 40 questions de test réservées.

#### *5.3.1. Définition du protocole et vérification de la couverture documentaire*

In [ ]:
import json
from collections import Counter

# 1. Fichier d'annotations validé
with qrels_path.open("r", encoding="utf-8") as file:
    evaluation_qrels = json.load(file)
annotations = evaluation_qrels["annotations"]

# 2. Vérif des questions de développement.
dev_ids = {item["id"] for item in dev_questions}

assert set(annotations) == dev_ids
assert len(annotations) == 10

# 3. Vérif des 55 jugements de chaque question.
for question_id, labels in annotations.items():
    assert set(labels) == all_chunk_ids
    assert set(labels.values()).issubset({0, 1, 2})

# 4. Définition des deux niveaux de pertinence.
relevance_thresholds = {"principal": 1, "strict": 2}

# 5. Calcul de la couverture pour chaque convention.
coverage_summary = {}

print("=== Couverture documentaire ===")

for evaluation_name, minimum_grade in relevance_thresholds.items():
    relevant_by_question = {}
    for question_id, labels in annotations.items():
        relevant_ids = {chunk_id for chunk_id, grade in labels.items() if grade >= minimum_grade}
        relevant_by_question[question_id] = relevant_ids

    covered_questions = sum(bool(relevant_ids) for relevant_ids in relevant_by_question.values())
    total_relevant = sum(len(relevant_ids) for relevant_ids in relevant_by_question.values())

    coverage_summary[evaluation_name] = {
        "covered_questions": covered_questions,
        "total_relevant": total_relevant,
        "relevant_by_question": relevant_by_question
    }

    print(f"\nÉvaluation : {evaluation_name}")
    print("Note minimale :", minimum_grade)
    print("Questions couvertes :", covered_questions, "/ 10")
    print("Passages pertinents (question-chunk) :", total_relevant)

# 6. Vérif des résultats attendus.
assert coverage_summary["principal"]["covered_questions"] == 8
assert coverage_summary["principal"]["total_relevant"] == 21
assert coverage_summary["strict"]["covered_questions"] == 7
assert coverage_summary["strict"]["total_relevant"] == 7

# 7. Identifier les questions non couvertes.
for evaluation_name, summary in coverage_summary.items():

    uncovered_ids = sorted(
        question_id
        for question_id, relevant_ids
        in summary["relevant_by_question"].items()
        if not relevant_ids
    )

    print(f"\nQuestions non couvertes ({evaluation_name}) :")
    print(uncovered_ids)


# 8. Vérifier la distribution globale des notes.
grade_counts = Counter(grade for labels in annotations.values() for grade in labels.values())
assert grade_counts == {0: 529, 1: 14, 2: 7}

# 9. Affichage du bilan.
print("\n=== Bilan des annotations ===")

print("Questions :", len(annotations))
print("Jugements :", sum(grade_counts.values()))
print("Notes 0 :", grade_counts[0])
print("Notes 1 :", grade_counts[1])
print("Notes 2 :", grade_counts[2])

print("\n[OK] Annotations complètes et cohérentes.")
print("[OK] Deux conventions de pertinence vérifiées.")
print("[OK] Questions sans passage pertinent identifiées.")
print("[OK] Prêt pour le calcul des métriques de retrieval.")

#### *5.3.2. Implémentation des métriques de retrieval*

**Objectif :** développer et valider les fonctions de calcul de Precision@k, Recall@k et du rang réciproque à partir des jugements documentaires.

**Méthodologie :**

1. Recevoir un classement de chunks.
2. Identifier les passages pertinents dans le Top-k.
3. Calculer Precision@k.
4. Calculer Recall@k si la question est couverte.
5. Déterminer le rang du premier passage pertinent.
6. Calculer son rang réciproque.
7. Tester les fonctions sur un exemple contrôlé.
8. Vérifier le cas d'une question non couverte.

**Conventions :**

- La pertinence est déterminée par le jeu de jugements, et non par les scores de similarité FAISS.
- Precision@k est calculée sur les k résultats.
- Recall@k est indéfini sans passage pertinent.
- Le rang réciproque est nul lorsqu'aucun pertinent n'apparaît dans le Top-k d'une question couverte.
- Le rang réciproque est indéfini lorsqu'aucun passage pertinent n'existe dans le corpus.
- MRR@k sera calculé en agrégeant les rangs réciproques des questions couvertes.

**Résultats attendus :**

- Une fonction de calcul réutilisable.
- Des métriques vérifiées sur un exemple connu.
- Une gestion explicite des questions non couvertes.
- Aucun changement des annotations ou de l'index.

**Point de vigilance :**

Cette première étape vérifie les formules. Les véritables performances du moteur seront calculées ensuite sur les dix questions de développement.

In [ ]:
# 1. Fonction de calcul pour une question
def compute_retrieval_metrics(ranked_chunk_ids, relevant_chunk_ids, top_k):
    # Vérifier le paramètre k.
    if type(top_k) is not int or top_k < 1:
        raise ValueError("top_k doit être un entier strictement positif.")

    # Vérifier que le classement contient k documents.
    if len(ranked_chunk_ids) < top_k:
        raise ValueError("Le classement contient moins de k documents.")

    # Vérifier que les identifiants sont uniques.
    if len(ranked_chunk_ids) != len(set(ranked_chunk_ids)):
        raise ValueError("Le classement contient des doublons.")

    # 2. Conservation uniquement des k premiers résultats.
    retrieved_ids = ranked_chunk_ids[:top_k]

    # 3. Identifiactaion des docs pertinents retrouvés
    relevant_ids = set(relevant_chunk_ids)
    retrieved_relevant = [chunk_id for chunk_id in retrieved_ids if chunk_id in relevant_ids]
    relevant_count = len(retrieved_relevant)
    total_relevant = len(relevant_ids)

    # 4. Precision@k
    precision = relevant_count / top_k

    # 5. Traitement des questions sans passage pertinent
    if total_relevant == 0:
        recall = None
        reciprocal_rank = None
        first_relevant_rank = None
    else:
        # 6. Rappel@k
        recall = relevant_count / total_relevant

        # 7. Recherche du premier doc pertinent
        first_relevant_rank = next((rank for rank, chunk_id in enumerate(retrieved_ids, start=1) if chunk_id in relevant_ids), None)

        # 8. Calcul du rang réciproque
        if first_relevant_rank is None:
            reciprocal_rank = 0.0
        else:
            reciprocal_rank = 1.0 / first_relevant_rank

    # 9. Retourner les résultats structurés
    return {
        "top_k": top_k,
        "retrieved_relevant": relevant_count,
        "total_relevant": total_relevant,
        "precision": precision,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "reciprocal_rank": reciprocal_rank
    }

# 10. Construction d'un classement fictif contrôlé
sample_ranking = ["chunk_A", "chunk_B", "chunk_C", "chunk_D", "chunk_E"]

# Quatre documents pertinents dans le corpus.
sample_relevant = {"chunk_B", "chunk_C", "chunk_F", "chunk_G"}

# 11. Métriques du Top-5
sample_metrics = compute_retrieval_metrics(
    ranked_chunk_ids=sample_ranking,
    relevant_chunk_ids=sample_relevant,
    top_k=5
)

# 12. Vérif des résultats mathématiques.
assert sample_metrics["retrieved_relevant"] == 2
assert sample_metrics["total_relevant"] == 4

assert sample_metrics["precision"] == 2 / 5
assert sample_metrics["recall"] == 2 / 4

assert sample_metrics["first_relevant_rank"] == 2
assert sample_metrics["reciprocal_rank"] == 1 / 2


# 13. Test du Top-1 sur la même question.
top_1_metrics = compute_retrieval_metrics(
    ranked_chunk_ids=sample_ranking,
    relevant_chunk_ids=sample_relevant,
    top_k=1
)

assert top_1_metrics["precision"] == 0.0
assert top_1_metrics["recall"] == 0.0
assert top_1_metrics["reciprocal_rank"] == 0.0

# 14. Test d'une question sans passage pertinent.
uncovered_metrics = compute_retrieval_metrics(
    ranked_chunk_ids=sample_ranking,
    relevant_chunk_ids=set(),
    top_k=5
)

assert uncovered_metrics["precision"] == 0.0
assert uncovered_metrics["recall"] is None
assert uncovered_metrics["reciprocal_rank"] is None


# 15. Afficher les résultats.
print("=== Vérification des métriques ===")

print("Documents pertinents dans le corpus : 4")
print("Documents pertinents retrouvés : 2")

print(f"\nPrecision@5 : {sample_metrics['precision']:.4f}")
print(f"Recall@5 : {sample_metrics['recall']:.4f}")
print("Premier rang pertinent :", sample_metrics["first_relevant_rank"])
print(f"Rang réciproque : {sample_metrics['reciprocal_rank']:.4f}")

print("\n=== Question non couverte ===")

print("Precision@5 :", uncovered_metrics["precision"])
print("Recall@5 :", uncovered_metrics["recall"])
print("Rang réciproque :", uncovered_metrics["reciprocal_rank"])

print("\n[OK] Precision@k validée.")
print("[OK] Recall@k validé.")
print("[OK] Rang réciproque validé.")
print("[OK] Cas sans passage pertinent pris en charge.")

#### *5.3.3. Évaluation sur les questions de développement*

**Objectif :** mesurer les performances réelles du moteur FAISS sur les dix questions annotées.

**Méthodologie :**

1. Effectuer une recherche Top-5 par question.
2. Conserver le classement des chunks retrouvés.
3. Calculer les métriques pour k = 1, 3 et 5.
4. Comparer les évaluations principale et stricte.
5. Calculer les moyennes sur les questions couvertes.
6. Rapporter séparément la précision globale.
7. Vérifier la cohérence des résultats numériques.

**Indicateurs :**

- Precision@k sur les dix questions.
- Precision@k sur les questions couvertes.
- Recall@k conditionnel.
- MRR@k conditionnel.
- Couverture documentaire.

**Résultats attendus :**

- Dix classements FAISS Top-5.
- Six configurations d'évaluation.
- Des métriques calculées à partir des jugements enregistrés dans `retrieval_qrels_dev.json`.
- Une comparaison entre pertinence partielle et pertinence directe.

**Points de vigilance :**

- Ne pas utiliser les 40 questions de test.
- Ne pas modifier les jugements humains.
- Ne pas ajuster les résultats FAISS.
- Ne pas interpréter la similarité cosinus comme une probabilité de pertinence.
- Ne pas confondre une question non couverte avec un échec de classement.

In [ ]:
# 1. Valeurs de k
eval_k_values = [1, 3, 5]
max_k = max(eval_k_values)

# 2. Effectuer une seule recherche Top-5 par question
dev_rankings = {}

print("=== Recherches FAISS sur le développement ===")
for item in dev_questions:
    question_id = item["id"]
    results = retrieve_documents(question=item["question"], top_k=max_k)
    ranked_ids = [result["chunk_id"] for result in results]

    # 3. Vérif des résultats retournés.
    assert len(ranked_ids) == max_k
    assert len(set(ranked_ids)) == max_k
    assert set(ranked_ids).issubset(all_chunk_ids)

    dev_rankings[question_id] = ranked_ids

    print(f"[OK] {question_id} : {len(ranked_ids)} documents retrouvés.")

# 4. Vérif des dix classements.
assert len(dev_rankings) == 10
assert set(dev_rankings) == {item["id"] for item in dev_questions}

# 5. Préparation des résultats quantitatifs.
development_metrics = {}

print("\n=== Évaluation quantitative du retrieval ===")

# 6. Parcours des deux conventions de pertinence.
for evaluation_name in relevance_thresholds:
    relevant_by_question = coverage_summary[evaluation_name]["relevant_by_question"]

    # Identifier les questions disposant d'au moins un passage pertinent.
    covered_ids = [question_id for question_id in dev_rankings if relevant_by_question[question_id]]
    assert len(covered_ids) == coverage_summary[evaluation_name]["covered_questions"]

    development_metrics[evaluation_name] = {}

    print(f"\n--- Évaluation : {evaluation_name} ---")
    print("Questions couvertes :", len(covered_ids), "/ 10")

    print("\nk | Precision globale | Precision couverte | Recall | MRR")

    # 7. Évaluer les différentes valeurs de k.
    for k in eval_k_values:
        question_metrics = {}
        for question_id, ranked_ids in (dev_rankings.items()):
            metrics = compute_retrieval_metrics(
                ranked_chunk_ids=ranked_ids,
                relevant_chunk_ids=relevant_by_question[question_id],
                top_k=k
            )

            question_metrics[question_id] = metrics

        # 8. Calcul de la précision globale.
        precision_all = sum(metrics["precision"] for metrics in question_metrics.values()) / len(question_metrics)

        # 9. Calcul des moyennes sur les questions couvertes uniquement.
        precision_covered = sum(question_metrics[question_id]["precision"] for question_id in covered_ids) / len(covered_ids)
        recall_covered = sum(question_metrics[question_id]["recall"] for question_id in covered_ids) / len(covered_ids)
        mrr_covered = sum(question_metrics[question_id]["reciprocal_rank"] for question_id in covered_ids) / len(covered_ids)

        # 10. Vérif des bornes des métriques.
        for value in [precision_all, precision_covered, recall_covered, mrr_covered]:
            assert 0.0 <= value <= 1.0

        # 11. Conservation les résultats.
        development_metrics[evaluation_name][k] = {
            "precision_all": precision_all,
            "precision_covered": precision_covered,
            "recall_covered": recall_covered,
            "mrr_covered": mrr_covered,
            "question_metrics": question_metrics
        }

        # 12. Affichage des scores.
        print(
            f"{k} | "
            f"{precision_all:.4f} | "
            f"{precision_covered:.4f} | "
            f"{recall_covered:.4f} | "
            f"{mrr_covered:.4f}"
        )

# 13. Vérif de la croissance du Recall et du MRR avec le nombre de docs retrouvés
for evaluation_name, results_by_k in (development_metrics.items()):
    recall_values = [results_by_k[k]["recall_covered"] for k in eval_k_values]
    mrr_values = [results_by_k[k]["mrr_covered"] for k in eval_k_values]
    assert recall_values == sorted(recall_values)
    assert mrr_values == sorted(mrr_values)

# 14. Afficher le bilan de l'expérience.
print("\n=== Bilan de l'évaluation ===")

print("Questions de développement :", len(dev_rankings))
print("Valeurs de k :", eval_k_values)

print("Configs évaluées :", len(relevance_thresholds) * len(eval_k_values))

print("\n[OK] Dix classements FAISS obtenus.")
print("[OK] Precision@k calculée.")
print("[OK] Recall@k conditionnel calculé.")
print("[OK] MRR@k conditionnel calculé.")
print("[OK] Cohérence des métriques vérifiée.")
print("[OK] Jeu de test réservé non utilisé.")

#### *5.3.4. Analyse des classements et des erreurs*

**Objectif :** examiner les performances du retrieval question par question, au-delà des moyennes globales.

**Méthodologie :**

1. Réutiliser les dix classements FAISS Top-5.
2. Associer à chaque résultat sa note humaine.
3. Identifier le rang du premier passage pertinent.
4. Comparer les évaluations principale et stricte.
5. Repérer les passages pertinents manqués.
6. Distinguer les erreurs de classement des questions non couvertes par le corpus.

**Résultats attendus :**

- Un bilan par question.
- Une lecture des cinq notes dans l'ordre FAISS.
- Les rangs des premiers passages pertinents.
- L'identification des questions à analyser plus précisément.

**Points de vigilance :**

- Ne pas modifier les annotations selon les scores.
- Ne pas confondre pertinence et similarité.
- Conserver les questions sans passage pertinent.
- Ne pas ajuster les paramètres sur le jeu de test.

In [ ]:
# 1. Récup des métriques Top-5 déjà calculées.
principal_metrics = development_metrics["principal"][5]["question_metrics"]
strict_metrics = development_metrics["strict"][5]["question_metrics"]

# 2. Affichage des résultats question par question.
print("=== Analyse des dix questions ===")
print("ID | Notes Top-5 | Pertinents retrouvés | Premier rang principal | Premier rang strict")

for item in dev_questions:
    question_id = item["id"]
    ranked_ids = dev_rankings[question_id]

    # 3. Pour retrouver les notes humaines des cinq passages.
    grades = [annotations[question_id][chunk_id] for chunk_id in ranked_ids]

    # 4. Récup les métriques de chaque convention.
    principal = principal_metrics[question_id]
    strict = strict_metrics[question_id]

    print(
        f"{question_id} | "
        f"{grades} | "
        f"{principal['retrieved_relevant']}/"
        f"{principal['total_relevant']} | "
        f"{principal['first_relevant_rank']} | "
        f"{strict['first_relevant_rank']}"
    )


# 5. Identification des questions couvertes dont le premier passage pertinent manque au Top-5.
print("\n=== Échecs de récupération Top-5 ===")

retrieval_failures = []

for question_id, metrics in principal_metrics.items():
    if (metrics["total_relevant"] > 0 and metrics["retrieved_relevant"] == 0):
        retrieval_failures.append(question_id)

print("Questions concernées :", retrieval_failures)

# 6. Identification des questions que le corpus ne couvre pas.
print("\n=== Limites de couverture documentaire ===")

uncovered_questions = [question_id for question_id, metrics in principal_metrics.items() if metrics["total_relevant"] == 0]
print("Questions non couvertes :", uncovered_questions)


# 7. Calcul du nombre de passages pertinents encore absents des Top-5.
total_relevant = sum(metrics["total_relevant"] for metrics in principal_metrics.values())
total_found = sum(metrics["retrieved_relevant"] for metrics in principal_metrics.values())
total_missed = total_relevant - total_found

print("\n=== Bilan des passages pertinents ===")

print("Passages pertinents dans les qrels :", total_relevant)
print("Passages retrouvés dans les Top-5 :", total_found)
print("Passages pertinents non retrouvés :", total_missed)


# 8. Vérif des résultats à partir des métriques globales.
assert total_relevant == 21
assert total_found == 12
assert total_missed == 9

assert uncovered_questions == ["CV009", "CV010"]

# 9. Vérif que les sept passages de niveau 2 sont tous retrouvés dans les Top-5.
strict_found = sum(metrics["retrieved_relevant"] for metrics in strict_metrics.values())
assert strict_found == 7


# 10. Affichage des confirmations.
print("\n[OK] Classements Top-5 analysés.")
print("[OK] Notes humaines associées aux résultats.")
print("[OK] Échecs de récupération identifiés.")
print("[OK] Limites du corpus distinguées.")
print("[OK] Aucun nouveau calcul FAISS nécessaire.")

### *5.4. Analyse qualitative des erreurs de classement*

**Objectif :** comprendre pourquoi certains passages pertinents ne figurent pas parmi les premiers résultats de recherche.

**Méthodologie :**

1. Sélectionner les questions CV001, CV002, CV007 et CV008.
2. Examiner les cinq premiers passages retournés.
3. Afficher leurs identifiants, sections et jugements de pertinence.
4. Lire des extraits des passages classés premiers et directement pertinents.
5. Identifier les passages pertinents non retrouvés.
6. Distinguer erreurs de classement et limites de couverture du corpus.

**Résultats attendus :**

- Une analyse des erreurs de classement de CV001, CV002 et CV007.
- Une analyse de la couverture partielle de CV008.
- Des observations exploitables pour choisir les futures configurations RAG.

**Points de vigilance :**

- Ne pas modifier les notes humaines.
- Ne pas modifier les embeddings ou l'index.
- Ne pas ajuster les paramètres sur le jeu de test.
- Ne pas conclure qu'un meilleur rappel garantit une meilleure qualité de réponse générée.


In [ ]:
# 1. Sélection des questions à examiner.
diagnostic_question_ids = ["CV001", "CV002", "CV007", "CV008"]

# 2. Examen des classements enregistrés.
for question_id in diagnostic_question_ids:
    question = development_by_id[question_id]
    ranked_ids = dev_rankings[question_id]
    labels = annotations[question_id]

    print("\n" + "=" * 70)
    print("Question :", question_id)
    print(question["question"])

    print("\n=== Classement Top-5 ===")

    # 3. Affichage des 5 passages avec leurs notes
    for rank, chunk_id in enumerate(ranked_ids, start=1):
        chunk = chunks_by_id[chunk_id]
        grade = labels[chunk_id]

        print(f"\n{rank}. {chunk_id} | note={grade} | section={chunk['section']}")

        # 4. Examen d'un extrait du premier résultata et de tout passage directement pertinent
        if rank == 1 or grade == 2:
            excerpt = " ".join(chunk["text"].split())
            print("   Extrait :", excerpt[:220])

    # 5. Identification des passages pertinents absents des cinq premiers résultats
    missing_relevant_ids = sorted(
        chunk_id 
        for chunk_id, grade in labels.items() 
        if grade >= 1 and chunk_id not in ranked_ids
    )

    print("\nPassages pertinents hors Top-5 :")
    if missing_relevant_ids:
        for chunk_id in missing_relevant_ids:
            print("-", chunk_id, "| note=", labels[chunk_id])
    else:
            print("Aucun")

# 6. Vérif du périmètre & des annotations
assert len(diagnostic_question_ids) == 4
assert all(question_id in dev_rankings and question_id in annotations for question_id in diagnostic_question_ids)

# 7. Affichage du bilan
print("\n=== Bilan du diagnostic ===")
print("Questions analysées :", len(diagnostic_question_ids))
print("Top-k examiné : 5")

print("\n[OK] Classements documentaires examinés.")
print("[OK] Jugements humains conservés.")
print("[OK] Passages manqués identifiés.")
print("[OK] Aucun nouvel appel FAISS effectué.")

## **6. Synthèse scientifique et limites**

**Objectif :** synthétiser la construction, la validation et l'évaluation du moteur de recherche documentaire avant son intégration à Qwen3-1.7B.

**Méthodologie :**

1. Récapituler la configuration du moteur.
2. Présenter les résultats quantitatifs mesurés.
3. Interpréter les erreurs de classement observées.
4. Identifier les limites du corpus et du protocole.
5. Préparer les futures expérimentations RAG.

### *6.1. Configuration évaluée*

Le moteur documentaire repose sur :

- Un corpus de **55 chunks** provenant de sept pages de documentation OpenCV.
- Le modèle `intfloat/multilingual-e5-small`.
- Des embeddings normalisés de **384 dimensions**.
- Le traitement de deux chunks longs par fenêtres de tokens et agrégation de leurs représentations.
- Un index FAISS exact `IndexFlatIP`.
- Une recherche sémantique avec traçabilité des sources.

L'évaluation utilise **10 questions de développement**, avec **550 jugements de pertinence** attribués aux couples question-chunk.

Les **40 questions de test réservées** n'ont pas été utilisées pour ajuster le moteur.

### *6.2. Résultats quantitatifs*

Deux conventions ont été étudiées :

- **Principale :** notes 1 et 2 considérées pertinentes.
- **Stricte :** seule la note 2 est considérée pertinente.

| Indicateur | Top-1 | Top-3 | Top-5 |
|---|---:|---:|---:|
| Precision globale — principale | 0,5000 | 0,3000 | 0,2400 |
| Recall macro conditionnel — principal | 0,3021 | 0,4583 | 0,6250 |
| MRR conditionnel — principal | 0,6250 | 0,7292 | 0,7604 |
| Precision globale — stricte | 0,4000 | 0,2000 | 0,1400 |
| Recall macro conditionnel — strict | 0,5714 | 0,8571 | 1,0000 |
| MRR conditionnel — strict | 0,5714 | 0,6905 | 0,7262 |

La couverture documentaire atteint :

- **80 %** avec la convention principale (8/10).
- **70 %** avec la convention stricte (7/10).

À k = 5, le moteur retrouve les **sept passages directement pertinents** existant dans le corpus pour les questions de développement.

En revanche, seulement **12 des 21 couples question-chunk pertinents** selon la convention principale sont retrouvés dans les Top-5.

Le rappel micro correspondant est donc de **12/21 = 57,14 %**, distinct de la moyenne
macro conditionnelle de 62,50 %.

### *6.3. Analyse qualitative*

Les diagnostics montrent plusieurs difficultés :

- **CV001 :** le passage sur l'ouverture morphologique apparaît au rang 2, derrière une présentation générale de la morphologie.
- **CV002 :** le passage sur la fermeture apparaît au rang 3, derrière des passages moins pertinents.
- **CV007 :** l'explication de l'hystérésis de Canny apparaît au rang 4 ; des passages concernant notamment le seuillage d'Otsu la précèdent.
- **CV008 :** un passage partiellement pertinent apparaît au premier rang, mais le corpus ne contient aucune réponse directement pertinente.

Ces observations indiquent que la similarité sémantique ne garantit pas toujours un classement optimal selon la pertinence documentaire humaine.

### *6.4. Limites scientifiques*

**Couverture documentaire :**

- CV008 ne possède aucun passage de note 2.
- CV009 et CV010 ne possèdent aucun passage de note 1 ou 2.
- Le moteur ne peut pas récupérer une explication absente des 55 passages indexés.

**Évaluation :**

- Les résultats reposent sur seulement dix questions.
- Les jugements ont été élaborés manuellement ; ils ne bénéficient pas d'une validation inter-annotateurs indépendante.
- Les métriques conditionnelles excluent les questions non couvertes et doivent être accompagnées des taux de couverture.
- Les résultats sur le développement ne constituent pas une estimation finale de généralisation.

**Représentations documentaires :**

- Deux chunks dépassant la limite de longueur de l'encodeur ont été représentés par une agrégation de fenêtres.
- L'effet de cette stratégie sur le classement n'a pas été isolé expérimentalement.

**Génération :**

- Aucune réponse RAG de Qwen3-1.7B n'a encore été évaluée.
- Un passage retrouvé ne garantit ni une réponse correcte, ni l'absence d'hallucinations.

### *6.5. Conclusion et perspectives*

Le pipeline **E5 + FAISS** est opérationnel, reproductible et évalué sur le jeu de développement.

Les résultats montrent une bonne capacité à retrouver les passages directement pertinents dans le Top-5 lorsqu'ils existent.

La précision décroissante lorsque k augmente, les passages partiellement pertinents manqués et les lacunes du corpus justifient cependant une évaluation du contexte réellement fourni au modèle génératif.

Pour la prochaine phase, **Top-3 et Top-5 seront des configurations candidates du RAG**, et non des choix définitivement optimaux.

L'objectif sera de comparer les réponses de Qwen3-1.7B avec et sans récupération documentaire, en conservant les sources et en contrôlant les erreurs factuelles.

Le jeu de test réservé restera distinct des analyses et choix effectués sur les dix questions de développement.
